# Whole-period CSS integration

Set the paths marked in the code before running this notebook.

Selected HVG2000, CSS reference resolution 0.4, neighbors 10, final Leiden 0.8.


In [ ]:
#1. =========== Paths and shared settings
# Replace each ROOT placeholder with the required file or directory path before running.
from pathlib import Path
import os
import sys

ROOT = Path(".")  # Set the project root before running this template.
RANDOM_SEED = 0
N_THREADS = 4
COUNTS_LAYER = 'counts'
STUDY_KEY = 'study_id'
BATCH_KEY = 'css_batch'
# Set the temporary working directory or file path.
TASK_TMP = ROOT

for key, leaf in [('NUMBA_CACHE_DIR', 'numba'), ('MPLCONFIGDIR', 'mpl'),
                  ('TMP', 'temp'), ('TEMP', 'temp')]:
    # Set the temporary working directory or file path.
    folder = ROOT
    folder.mkdir(parents=True, exist_ok=True)
    os.environ[key] = str(folder)
for key in ('OMP_NUM_THREADS', 'MKL_NUM_THREADS', 'OPENBLAS_NUM_THREADS',
            'NUMBA_NUM_THREADS'):
    os.environ[key] = str(N_THREADS)
sys.dont_write_bytecode = True

import hashlib
from importlib.metadata import version
import h5py
import numpy as np
import pandas as pd
import anndata as ad
import scanpy as sc
from anndata.io import read_elem
from scipy import sparse
from sklearn.utils.sparsefuncs import mean_variance_axis
from threadpoolctl import threadpool_limits
from IPython.display import display

sc.settings.verbosity = 0
sc.settings.n_jobs = N_THREADS
package_versions = {name: version(name) for name in
                    ('scanpy', 'anndata', 'numpy', 'scipy', 'pandas', 'scikit-learn',
                     'igraph', 'leidenalg', 'umap-learn', 'pynndescent', 'matplotlib')}

def file_sha256(path):
    digest = hashlib.sha256()
    with path.open('rb') as stream:
        for block in iter(lambda: stream.read(4 * 1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

def counts_sha256(matrix, cells, genes):
    digest = hashlib.sha256()
    for names in (cells, genes):
        digest.update(('\n'.join(map(str, names)) + '\n').encode())
    for values in (np.asarray(matrix.shape, dtype=np.int64),
                   matrix.indptr.astype(np.int64), matrix.indices.astype(np.int64),
                   matrix.data):
        digest.update(np.ascontiguousarray(values).tobytes())
    return digest.hexdigest()

In [ ]:
#2. =========== Fixed stage inputs
# Set the stage input directory; uses integration H5ADs generated by the corresponding fig1_E*_clustering.ipynb notebook.
EMBRYO_DIR = ROOT
# Set the data file or directory path.
POSTNATAL_DIR = ROOT
INPUTS = [
    ('E9', 12, 11467, '8c730404120c41ad725dd2ec8f5624b51481d9c8c73148599c937004b3bfc408'),
    ('E10', 73, 11467, 'f1a62ad282a54eda98e21bb394403b22f9490982aae28beea6923738f55ce3aa'),
    ('E11', 331, 11467, 'c8b869fd7196b0e54c6e65d67c760c5b334386a5561d104ab4ca7c87d7932a5e'),
    ('E13', 818, 11467, '4d7d01c2949f8e9f4fcbf4fedcae5a246c75c1258267f468826c47418ba0005f'),
    ('E14', 1674, 11467, '9dab7bdaf335db30969239fd3ca535281404d95fa20f7ff88139a0c92285945c'),
    ('E15', 161, 11467, '74e5be0e62f87563ebdcb31bbb85faf0c9ec5be7722fbb77f4f0f42a8b797196'),
    ('E16', 3028, 11467, '090a5c2cb7f4d8bb8a698a0feed503a6d784a434216e4c7ac3e1ca9c827cade5'),
    ('E17', 71, 11467, '4c6757bea906cad57b84022cbc5c83686348ee3dd24b9a449837d964398b823e'),
    ('E18', 1187, 11467, 'ffff655ce3d282b701c57e8ba18410a274f7a8c742f431b1a79b1d032e433ace'),
    ('P1', 314, 13237, '55c39dbcb17d7c5c6e75f9d967eff236c010832054f9d4cec0b10d14aab19b10'),
    ('P7', 387, 29213, '7ae87aa77e6d9bbd4585f829b9da1d0d4cef60dfad44a3e4614bbe81615f8e25'),
    ('P28', 284, 18019, 'c57303a29aa345d648251a25628cf3b3339b7476f072a4fd4c212a908b0cffc2'),
]

parts, records = {}, []
for stage, n_cells, n_genes, expected_sha in INPUTS:
    embryo = stage.startswith('E')
    # Set each stage input path; uses {stage}_Embryonic_Inte_adata.h5ad generated by the corresponding fig1_E*_clustering.ipynb notebook.
    # Set the stage result path; uses {stage}_cluster_intergration.h5ad generated by P1, P7 or P28 clustering.
    path = ((ROOT) if embryo else
            (ROOT))
    assert file_sha256(path) == expected_sha, f'Input changed: {path.name}'
    part = ad.read_h5ad(path)
    assert part.shape == (n_cells, n_genes), f'Unexpected shape: {stage}'
    assert part.obs_names.is_unique and part.var_names.is_unique, stage
    assert COUNTS_LAYER in part.layers, f'Missing counts: {stage}'
    counts = sparse.csr_matrix(part.layers[COUNTS_LAYER], copy=True)
    counts.sum_duplicates()
    counts.sort_indices()
    assert np.issubdtype(counts.dtype, np.integer), f'Noninteger counts: {stage}'
    assert np.isfinite(counts.data).all() and (counts.data >= 0).all(), stage
    assert (np.asarray(counts.sum(axis=1)).ravel() > 0).all(), stage
    assert (sparse.csr_matrix(part.X) != counts).nnz == 0, f'X is not raw counts: {stage}'
    part.layers[COUNTS_LAYER] = counts
    assert part.obs[STUDY_KEY].notna().all(), f'Missing study: {stage}'
    source_batch = 'harmony_batch_used' if embryo else 'integration_batch'
    assert part.obs[source_batch].notna().all(), f'Missing batch: {stage}'
    part.obs[BATCH_KEY] = part.obs[source_batch].astype(str)
    part.obs['input_stage'] = f'{stage}.5' if embryo else stage
    part.obs['input_file'] = path.name
    if 'qc_pass' in part.obs:
        assert part.obs['qc_pass'].notna().all() and part.obs['qc_pass'].all(), stage
    for field in [c for c in part.obs if c.startswith('fail_')]:
        assert not part.obs[field].eq(True).any(), f'Source QC failure: {stage}/{field}'
    if 'selected_for_integration' in part.obs:
        assert part.obs['selected_for_integration'].all(), stage
    if 'predicted_doublet' in part.obs:
        assert not part.obs['predicted_doublet'].eq(True).any(), stage
    if 'blood_qc_pass' in part.obs:
        assert part.obs['blood_qc_pass'].notna().all() and part.obs['blood_qc_pass'].all(), stage
    parts[stage] = part
    records.append(dict(stage=stage, path=str(path), sha256=expected_sha,
                        cells=n_cells, source_genes=n_genes,
                        counts_sha256=counts_sha256(counts, part.obs_names, part.var_names)))

input_manifest = pd.DataFrame(records).set_index('stage')
cell_order = pd.Index(np.concatenate([p.obs_names.to_numpy() for p in parts.values()]))
assert len(cell_order) == 8340 and cell_order.is_unique, 'Cell IDs overlap across inputs.'

In [ ]:
#3. =========== Verified reference mapping
# Set the data file path; uses postnatal_prepared.h5ad generated by fig1_Postnatal_preparation.py.
MAPPING_FILE = ROOT
MAPPING_SHA256 = 'a5855620b364fbfde700142db82d3d04e82cc20e0d89d648b85201b206fb4365'

assert file_sha256(MAPPING_FILE) == MAPPING_SHA256, 'Mapping file changed.'
with h5py.File(MAPPING_FILE, 'r') as handle:
    mapping_var = read_elem(handle['var'])
    gene_reference = read_elem(handle['uns/gene_reference'])
embryo_var = parts['E9'].var.copy()
assert mapping_var.index.is_unique and len(mapping_var) == 10242
assert np.array_equal(embryo_var.index.astype(str), embryo_var.canonical_gene_id.astype(str))
for stage, part in parts.items():
    if stage.startswith('E'):
        assert part.var_names.equals(embryo_var.index), f'Embryo axis changed: {stage}'
        assert np.array_equal(part.var.author_glial_common_excluded936,
                              embryo_var.author_glial_common_excluded936), stage


In [ ]:
#4. =========== Restore verified postnatal-zero genes
RESTORED_GENES = [
    ('ENSMUSG00000018930', 'Ccl4', 19),
    ('ENSMUSG00000019828', 'Grm1', 107),
    ('ENSMUSG00000020357', 'Flt4', 653),
    ('ENSMUSG00000021363', 'Mak', 762),
    ('ENSMUSG00000025955', 'Akr1cl', 143),
    ('ENSMUSG00000026807', 'Ak8', 839),
    ('ENSMUSG00000030406', 'Gipr', 1928),
    ('ENSMUSG00000030670', 'Cyp2r1', 342),
    ('ENSMUSG00000032083', 'Apoa1', 196),
    ('ENSMUSG00000037166', 'Ppp1r14a', 23306),
    ('ENSMUSG00000040046', 'Tph1', 2010),
    ('ENSMUSG00000040258', 'Nxph4', 418),
    ('ENSMUSG00000042604', 'Kcna4', 763),
    ('ENSMUSG00000050050', 'Ccdc158', 317),
    ('ENSMUSG00000058488', 'Kl', 905),
    ('ENSMUSG00000062687', 'Cks1brt', 114),
    ('ENSMUSG00000068744', 'Psrc1', 3261),
    ('ENSMUSG00000097665', 'Gm16853', 307),
]
verified = pd.DataFrame(
    RESTORED_GENES,
    columns=['gene_id', 'gene_symbol', 'embryo_counts']
).set_index('gene_id')

assert file_sha256(MAPPING_FILE) == MAPPING_SHA256
with h5py.File(MAPPING_FILE, 'r') as handle:
    mapping_var = read_elem(handle['var'])

old_genes = embryo_var.index.intersection(mapping_var.index, sort=False)
assert len(old_genes) == 9156 and len(mapping_var) == 10242
assert not verified.index.isin(mapping_var.index).any()
assert np.array_equal(
    embryo_var.loc[verified.index, 'gene_symbol'].astype(str),
    verified.gene_symbol
)
assert embryo_var.loc[verified.index, 'target_biotype'].astype(str).isin(
    ['protein_coding', 'antisense']
).all()

extra = verified[['gene_symbol']].copy()
flags = ['mt_feature', 'ERCC_feature', 'technical_feature', 'rRNA_feature']
for flag in flags:
    extra[flag] = False

for stage in ('P1', 'P7', 'P28'):
    part = parts[stage]
    source = part.var[
        'gene_symbol' if stage == 'P1' else 'gene_id_original'
    ].astype(str)
    unique = ~source.duplicated(keep=False)
    lookup = pd.Series(np.arange(part.n_vars)[unique], index=source[unique])
    wanted = verified.gene_symbol if stage == 'P1' else verified.index.to_series()
    idx = wanted.map(lookup)

    assert idx.notna().all(), f'Missing or duplicated source columns: {stage}'
    idx = idx.to_numpy(dtype=np.int64)
    source_var = part.var.iloc[idx]
    extra[f'{stage}_source_column'] = idx
    extra[f'{stage}_source_symbol'] = source_var.gene_symbol.to_numpy()
    extra[f'{stage}_source_gene_id'] = source_var.gene_id_original.to_numpy()

    for flag in ('mt_feature', 'ERCC_feature', 'technical_feature'):
        assert source_var[flag].notna().all(), f'Missing feature flags: {stage}/{flag}'
        extra[flag] |= source_var[flag].to_numpy(dtype=bool)

    assert part.layers[COUNTS_LAYER][:, idx].nnz == 0, stage

assert not extra[flags].any().any()
axis_mapping = pd.concat([mapping_var, extra], axis=0, sort=False)
common_genes = embryo_var.index.intersection(axis_mapping.index, sort=False)
assert len(common_genes) == 9174 and axis_mapping.index.is_unique

positions, blocks, retention = {}, [], []
for stage, part in parts.items():
    if stage.startswith('E'):
        idx = part.var_names.get_indexer(common_genes)
    else:
        rows = axis_mapping.loc[common_genes]
        idx = rows[f'{stage}_source_column'].to_numpy(dtype=np.int64)
        assert np.array_equal(
            part.var_names[idx].astype(str),
            rows[f'{stage}_source_symbol'].astype(str)
        ), stage
        assert np.array_equal(
            part.var.iloc[idx].gene_id_original.astype(str),
            rows[f'{stage}_source_gene_id'].astype(str)
        ), stage

    assert (idx >= 0).all() and len(np.unique(idx)) == len(common_genes), stage
    positions[stage] = idx
    block = part.layers[COUNTS_LAYER][:, idx].tocsr()
    blocks.append(block)
    source_total = np.asarray(part.layers[COUNTS_LAYER].sum(axis=1)).ravel()
    common_total = np.asarray(block.sum(axis=1)).ravel()
    assert (common_total > 0).all(), stage
    retention.append(pd.Series(common_total / source_total, index=part.obs_names))

joint_counts = sparse.vstack(blocks, format='csr', dtype=np.int32)
joint_counts.sum_duplicates()
joint_counts.sort_indices()

old_idx = common_genes.get_indexer(old_genes)
old_counts = sparse.vstack([
    part.layers[COUNTS_LAYER][:, (
        part.var_names.get_indexer(old_genes) if stage.startswith('E') else
        mapping_var.loc[old_genes, f'{stage}_source_column'].to_numpy(dtype=np.int64)
    )]
    for stage, part in parts.items()
], format='csr')
assert (joint_counts[:, old_idx] != old_counts).nnz == 0

new_idx = common_genes.get_indexer(verified.index)
embryo_n = sum(p.n_obs for stage, p in parts.items() if stage.startswith('E'))
assert joint_counts[embryo_n:, new_idx].nnz == 0
assert np.array_equal(
    np.asarray(joint_counts[:embryo_n, new_idx].sum(axis=0)).ravel(),
    verified.embryo_counts.to_numpy()
)

var = embryo_var.loc[common_genes].drop(
    columns=['highly_variable', 'means', 'dispersions', 'dispersions_norm'],
    errors='ignore'
).copy()
var[flags] = axis_mapping.loc[common_genes, flags].astype(bool)
technical = var.gene_symbol.astype(str).str.match(r'(?i)^mt-|^ERCC[-_]')
var['feature_excluded'] = (
    var.author_glial_common_excluded936.astype(bool)
    | technical
    | var[flags].any(axis=1)
)
assert var.feature_excluded.sum() == 803

obs = pd.concat([p.obs for p in parts.values()], axis=0, sort=False).copy()
obs['common_count_fraction'] = pd.concat(retention).reindex(obs.index)
a = ad.AnnData(X=joint_counts.copy(), obs=obs, var=var)
a.layers[COUNTS_LAYER] = joint_counts.copy()
assert a.obs_names.equals(cell_order) and a.shape == (8340, 9174)
assert (np.asarray(joint_counts.sum(axis=0)).ravel() > 0).all()

joint_counts_hash = counts_sha256(joint_counts, a.obs_names, a.var_names)
a.uns['input_manifest'] = input_manifest.reset_index().to_dict('list')
a.uns['package_versions'] = package_versions
a.uns['source_contracts'] = {stage: dict(part.uns) for stage, part in parts.items()}
a.uns['mapping_source'] = dict(
    path=str(MAPPING_FILE),
    sha256=MAPPING_SHA256,
    expression_used=False,
    reference=gene_reference,
    axis_scope='Ensembl115_with_verified_postnatal_zero_genes'
)
a.uns['restored_gene_mapping'] = dict(
    release=115,
    reference_base='https://ftp.ensembl.org/pub/release-115/mysql/ensembl_mart_115/',
    postnatal_common_10269_id_sha256='ced7024770a3ba04f882437e245ca63b31302e0ef668f3e129c2249af2220f4f',
    gene_main_sha256='265f76d9737f8398a6776179601f7be3db3391404ae993a17fa53591e816cb79',
    synonym_sha256='a0372585a01435cb592be20977bddfdceee04eee419c3efdd2468f9e34ca25d9',
    restored_ids=verified.index.to_numpy(dtype=str),
    validation='Complete Ensembl115 names and aliases; unique source mapping; original 10269-ID axis reproduced'
)

preparation_summary = pd.DataFrame([dict(
    cells=a.n_obs,
    common_genes=a.n_vars,
    restored_genes=len(verified),
    excluded_features=int(var.feature_excluded.sum()),
    feature_candidates=int((~var.feature_excluded).sum())
)])
display(preparation_summary)
del old_counts

In [ ]:
#5. =========== Source QC and blood-lineage review
BLOOD_MIN_MARKERS = 2
NEURAL_MIN_MARKERS = 3
EARLY_MIN_MARKERS = 2
PANELS = {
    'immune': ['Ptprc', 'Lyz2', 'Tyrobp', 'Fcer1g', 'C1qa', 'C1qb', 'Spi1'],
    'erythroid': ['Gypa', 'Klf1', 'Gata1', 'Alas2', 'Slc4a1'],
    'hemoglobin': ['Hba-a1', 'Hbb-bs'],
    'neuron': ['Tubb3', 'Nefl', 'Nefm', 'Elavl3', 'Elavl4', 'Stmn2',
               'Map2', 'Gap43', 'Uchl1', 'Ncam1'],
    'early_otic': ['Six1', 'Eya1', 'Isl1', 'Pou4f1', 'Fgf10', 'Neurog1', 'Neurod1', 'Insm1'],
    'glia': ['Sox10', 'Plp1', 'Mpz', 'S100b', 'Fabp7'],
}

markers = list(dict.fromkeys(gene for genes in PANELS.values() for gene in genes))
profiles, summaries = [], []
for stage, part in parts.items():
    values = np.full((part.n_obs, len(markers)), np.nan, dtype=np.float64)
    if stage.startswith('E'):
        raw = np.asarray(part.obsm['marker_raw_counts'], dtype=np.float64)
        measured = np.asarray(part.obsm['marker_measured'], dtype=bool)
        assert np.array_equal(np.isfinite(raw), measured), stage
        source_symbols = pd.Index(part.uns['marker_gene_symbols'].astype(str))
        denominator = part.obs.marker_common_total_counts.to_numpy(dtype=np.float64)
        assert source_symbols.is_unique, stage
        for j, gene in enumerate(markers):
            if gene in source_symbols:
                values[:, j] = raw[:, source_symbols.get_loc(gene)]
    else:
        source_symbols = part.var.gene_symbol.astype(str)
        denominator = np.asarray(part.layers[COUNTS_LAYER].sum(axis=1)).ravel().astype(np.float64)
        for j, gene in enumerate(markers):
            columns = np.flatnonzero(source_symbols.to_numpy() == gene)
            if len(columns):
                values[:, j] = np.asarray(part.layers[COUNTS_LAYER][:, columns].sum(axis=1)).ravel()
    assert np.isfinite(denominator).all() and (denominator > 0).all(), stage
    normalized = np.log1p(values * 10000 / denominator[:, None])
    review = pd.DataFrame(index=part.obs_names)
    review['stage'] = part.obs.input_stage
    for panel, genes in PANELS.items():
        idx = [markers.index(gene) for gene in genes]
        measured = np.isfinite(values[:, idx]).sum(axis=1)
        detected = (values[:, idx] > 0).sum(axis=1)
        average = np.divide(np.nansum(normalized[:, idx], axis=1), measured,
                            out=np.full(part.n_obs, np.nan), where=measured > 0)
        review[f'{panel}_measured'] = measured
        review[f'{panel}_detected'] = detected
        review[f'{panel}_mean_log'] = average
    review['blood_marker_signal'] = ((review.immune_detected >= BLOOD_MIN_MARKERS) |
                                     (review.erythroid_detected >= BLOOD_MIN_MARKERS))
    review['neural_support'] = ((review.neuron_detected >= NEURAL_MIN_MARKERS) |
                                (review.early_otic_detected >= EARLY_MIN_MARKERS))
    blood_mean = review[['immune_mean_log', 'erythroid_mean_log']].max(axis=1)
    neural_mean = review[['neuron_mean_log', 'early_otic_mean_log']].max(axis=1)
    review['requires_review'] = review.blood_marker_signal & (
        ~review.neural_support | (blood_mean >= neural_mean))
    for field in ('pct_mito', 'qc_pass', 'predicted_doublet', 'doublet_score',
                  'doublet_status', 'doublet_review', 'source_review_flag', 'identity_review',
                  'blood_lineage_status', 'source_rna_mixture_review_flag'):
        if field in part.obs:
            review[field] = part.obs[field]
    profiles.append(review)
    summaries.append(dict(
        stage=part.obs.input_stage.iloc[0], cells=part.n_obs, source_genes=part.n_vars,
        common_count_fraction_median=a.obs.loc[part.obs_names, 'common_count_fraction'].median(),
        immune_signal=(review.immune_detected >= BLOOD_MIN_MARKERS).sum(),
        erythroid_signal=(review.erythroid_detected >= BLOOD_MIN_MARKERS).sum(),
        immune_measured_min=review.immune_measured.min(),
        erythroid_measured_min=review.erythroid_measured.min(),
        hemoglobin_measured_min=review.hemoglobin_measured.min(),
        requires_review=review.requires_review.sum()))
marker_review = pd.concat(profiles).reindex(cell_order)
qc_candidates = marker_review.loc[marker_review.blood_marker_signal].copy()
qc_summary = pd.DataFrame(summaries).set_index('stage')
display(qc_summary.round({'common_count_fraction_median': 3}))
unresolved = marker_review.loc[marker_review.requires_review]
if len(unresolved):
    display(unresolved)
assert unresolved.empty, 'Blood-lineage marker candidates need review before proceeding.'
a.obs = a.obs.copy()
a.obs['blood_marker_signal'] = marker_review.blood_marker_signal
a.obs['blood_marker_neural_support'] = marker_review.neural_support
a.uns['qc_review'] = dict(screen_positive_cells=len(qc_candidates),
                         unresolved_marker_candidates=len(unresolved),
                         new_cells_removed=0,
                         scope='source_QC_and_blood_marker_review')

In [ ]:
#6. =========== Common-axis normalization
TARGET_SUM = 10000

a.X = a.layers[COUNTS_LAYER].astype(np.float32)
sc.pp.normalize_total(a, target_sum=TARGET_SUM)
sc.pp.log1p(a)
assert np.isfinite(a.X.data).all()
a.uns['X_definition'] = 'log1p(CP10k) on the complete verified common axis'
a.uns['preprocessing'] = dict(target_sum=TARGET_SUM, normalization_genes=a.n_vars,
                              counts_layer=COUNTS_LAYER, seed=RANDOM_SEED,
                              n_threads=N_THREADS)
assert counts_sha256(a.layers[COUNTS_LAYER], a.obs_names, a.var_names) == joint_counts_hash

In [ ]:
#7. =========== Within-study HVG ranking
HVG_COUNTS = (2000,)
HVG_RANK_N = 5000
HVG_FLAVOR = 'seurat'
HVG_N_BINS = 20

work = ad.AnnData(X=a.X.copy(), obs=a.obs[[STUDY_KEY]].copy(),
                 var=pd.DataFrame(index=a.var_names.copy()))
with threadpool_limits(limits=N_THREADS):
    sc.pp.highly_variable_genes(work, n_top_genes=HVG_RANK_N, flavor=HVG_FLAVOR,
                               n_bins=HVG_N_BINS, batch_key=STUDY_KEY)
    _, variance = mean_variance_axis(a.X, axis=0)
rank = work.var.assign(gene=work.var_names)
eligible = (~a.var.feature_excluded.to_numpy() & (variance > 0) &
            np.isfinite(variance) & np.isfinite(rank.dispersions_norm.to_numpy()) &
            np.isfinite(rank.means.to_numpy()))
rank = rank.loc[eligible].sort_values(
    ['highly_variable_nbatches', 'dispersions_norm', 'gene'],
    ascending=[False, False, True], kind='stable')
assert len(rank) >= max(HVG_COUNTS), f'Only {len(rank)} valid feature candidates.'
hvg_genes = {n: rank.index[:n].astype(str).tolist() for n in HVG_COUNTS}
a.uns['hvg_parameters'] = dict(flavor=HVG_FLAVOR, rank_n_top_genes=HVG_RANK_N,
                               n_bins=HVG_N_BINS, batch_key=STUDY_KEY,
                               eligible_features=len(rank), hvg_counts=list(HVG_COUNTS))
del work

In [ ]:
#8. =========== Matched PCA controls
N_PCS = 30
SCALE_CLIP = 10
PCA_SOLVER = 'arpack'

branches, pca_rows = {}, []
with threadpool_limits(limits=N_THREADS):
    for n_hvg in HVG_COUNTS:
        genes = hvg_genes[n_hvg]
        assert N_PCS < min(n_hvg, a.n_obs)
        z = a[:, genes].X.toarray().astype(np.float32, copy=False)
        mean = z.mean(axis=0)
        scale = np.maximum(z.std(axis=0, ddof=1), 1e-8)
        scaled = np.clip((z - mean) / scale, -SCALE_CLIP, SCALE_CLIP)
        pca = ad.AnnData(X=scaled, var=pd.DataFrame(index=pd.Index(genes)))
        sc.tl.pca(pca, n_comps=N_PCS, zero_center=True, svd_solver=PCA_SOLVER,
                  random_state=RANDOM_SEED, mask_var=None)
        assert pca.varm['PCs'].shape == (n_hvg, N_PCS)
        assert np.isfinite(pca.obsm['X_pca']).all()
        branch = a.copy()
        branch.var['highly_variable'] = branch.var_names.isin(genes)
        branch.obsm['X_matched_pca'] = pca.obsm['X_pca'].astype(np.float32).copy()
        branch.uns['ordered_genes'] = np.asarray(genes, dtype=str)
        branch.uns['pca_parameters'] = dict(
            n_hvg=n_hvg, n_comps=N_PCS, svd_solver=PCA_SOLVER,
            unit_variance=True, scale_ddof=1, scale_clip=SCALE_CLIP,
            mask_var=None, random_state=RANDOM_SEED,
            variance_ratio=pca.uns['pca']['variance_ratio'].copy())
        branches[n_hvg] = branch
        pca_rows.append(dict(HVG=n_hvg, cells=branch.n_obs,
                             common_genes=branch.n_vars, PCs=N_PCS,
                             variance_explained=pca.uns['pca']['variance_ratio'].sum()))
        del z, scaled, pca

In [ ]:
#9. =========== Preparation checks
for stage, part in parts.items():
    record = input_manifest.loc[stage]
    assert file_sha256(Path(record.path)) == record.sha256, f'Input changed: {stage}'
    assert counts_sha256(part.layers[COUNTS_LAYER], part.obs_names,
                          part.var_names) == record.counts_sha256, stage
assert file_sha256(MAPPING_FILE) == MAPPING_SHA256
assert a.shape == (8340, 9174)
assert a.obs[BATCH_KEY].notna().all() and a.obs[STUDY_KEY].notna().all()
assert a.obs[BATCH_KEY].nunique() == 38 and a.obs[STUDY_KEY].nunique() == 6
assert a.obs.groupby(BATCH_KEY, observed=True).size().min() >= 3
assert set(branches) == set(HVG_COUNTS)
for n_hvg, branch in branches.items():
    assert branch.obs_names.equals(cell_order) and branch.var_names.equals(a.var_names)
    assert counts_sha256(branch.layers[COUNTS_LAYER], branch.obs_names,
                          branch.var_names) == joint_counts_hash
    assert len(branch.uns['ordered_genes']) == n_hvg
    assert branch.obsm['X_matched_pca'].shape == (a.n_obs, N_PCS)
    assert 'X_css' not in branch.obsm and not branch.obsp
a.uns['preparation_complete'] = True
preparation_summary = pd.DataFrame(pca_rows).set_index('HVG')
preparation_summary['feature_candidates'] = len(rank)
display(preparation_summary.round({'variance_explained': 4}))

## PCA control

HVG2000: unintegrated PCA by stage and study, with variance explained.


In [ ]:
#10. =========== PCA diagnostics
PCA_FIGSIZE = (10.8, 4.8)
PCA_DISPLAY_DPI = 120
PCA_POINT_SIZE = 1.35
PCA_ALPHA = 0.82
PCA_LEGEND_FONT = 6.7

import colorsys
from io import BytesIO
from sklearn.neighbors import NearestNeighbors
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.colors import to_hex, to_rgb
from matplotlib.lines import Line2D
from IPython.display import Image

STAGE_ORDER = ('E9.5', 'E10.5', 'E11.5', 'E13.5', 'E14.5', 'E15.5',
               'E16.5', 'E17.5', 'E18.5', 'P1', 'P7', 'P28')

def category_contacts(matrix, codes, n_categories):
    codes = np.asarray(codes, dtype=np.int64)
    assert ((codes >= 0) & (codes < n_categories)).all()
    edges = matrix.tocoo()
    result = np.bincount(codes[edges.row] * n_categories + codes[edges.col],
                         weights=edges.data,
                         minlength=n_categories ** 2).reshape(n_categories, n_categories)
    np.fill_diagonal(result, 0)
    return result

def category_palette(categories, contacts, counts, fixed=None):
    palette = {} if fixed is None else {key: value for key, value in fixed.items()
                                       if key in categories}
    index = {key: i for i, key in enumerate(categories)}
    pool = [to_hex(colorsys.hls_to_rgb((i * 0.61803398875 + 0.13) % 1,
                                     0.55 + 0.035 * (i % 3), 0.48))
            for i in range(max(64, 2 * len(categories)))]
    pool = [color for color in pool if color not in palette.values()]
    for key in sorted(categories, key=lambda key: (-counts[index[key]], index[key])):
        if key in palette:
            continue
        if not palette:
            chosen = pool[0]
        else:
            assigned = list(palette)
            reference = np.array([to_rgb(palette[name]) for name in assigned])
            weights = contacts[index[key], [index[name] for name in assigned]]
            weights = weights / max(float(weights.max()), 1)
            candidates = np.array([to_rgb(color) for color in pool])
            distance = np.sqrt(((candidates[:, None] - reference[None]) ** 2
                                * np.array([2, 4, 3])).sum(axis=2))
            score = (distance / (0.15 + weights[None])).min(axis=1)
            chosen = pool[int(score.argmax())]
        palette[key] = chosen
        pool.remove(chosen)
    return palette

stages = a.obs.input_stage.astype(str).to_numpy()
studies = a.obs[STUDY_KEY].astype(str).to_numpy()
study_order = list(pd.unique(studies))
stage_codes = pd.Categorical(stages, categories=STAGE_ORDER).codes
study_codes = pd.Categorical(studies, categories=study_order).codes
assert (stage_codes >= 0).all() and (study_codes >= 0).all()
stage_contacts = np.zeros((len(STAGE_ORDER), len(STAGE_ORDER)))
study_contacts = np.zeros((len(study_order), len(study_order)))
for n_hvg in HVG_COUNTS:
    xy = branches[n_hvg].obsm['X_matched_pca'][:, :2]
    neighbors = NearestNeighbors(n_neighbors=10, n_jobs=N_THREADS).fit(xy)
    edges = neighbors.kneighbors(return_distance=False)
    rows = np.repeat(np.arange(a.n_obs), edges.shape[1])
    matrix = sparse.csr_matrix((np.ones(len(rows)), (rows, edges.ravel())),
                               shape=(a.n_obs, a.n_obs))
    matrix = matrix.maximum(matrix.T)
    stage_contacts += category_contacts(matrix, stage_codes, len(STAGE_ORDER))
    study_contacts += category_contacts(matrix, study_codes, len(study_order))
stage_palette = category_palette(STAGE_ORDER, stage_contacts,
                                  np.bincount(stage_codes, minlength=len(STAGE_ORDER)))
study_palette = category_palette(study_order, study_contacts,
                                  np.bincount(study_codes, minlength=len(study_order)))
draw_order = np.random.default_rng(RANDOM_SEED).permutation(a.n_obs)

for n_hvg in HVG_COUNTS:
    branch = branches[n_hvg]
    xy = branch.obsm['X_matched_pca'][:, :2]
    ratio = branch.uns['pca_parameters']['variance_ratio']
    fig, axes = plt.subplots(1, 3, figsize=PCA_FIGSIZE, facecolor='white')
    for ax, values, order, palette, title in (
        (axes[0], stages, STAGE_ORDER, stage_palette, 'Stage'),
        (axes[1], studies, study_order, study_palette, 'Study'),
    ):
        ax.scatter(xy[draw_order, 0], xy[draw_order, 1],
                   c=[palette[key] for key in values[draw_order]],
                   s=PCA_POINT_SIZE, alpha=PCA_ALPHA, linewidths=0)
        ax.set(xlabel='PC1', ylabel='PC2', title=title)
        ax.set_box_aspect(1)
        handles = [Line2D([], [], marker='o', linestyle='', color=palette[key],
                          markersize=4, label=key.replace('_', ' ')) for key in order]
        ax.legend(handles=handles, loc='upper center', bbox_to_anchor=(0.5, -0.23),
                  ncol=4 if title == 'Stage' else 2,
                  fontsize=PCA_LEGEND_FONT, frameon=False)
    pcs = np.arange(1, len(ratio) + 1)
    axes[2].plot(pcs, ratio * 100, color='#6874A2', marker='o', markersize=2)
    axes[2].set(xlabel='PC', ylabel='Variance explained (%)', title='Scree plot')
    cumulative = axes[2].twinx()
    cumulative.plot(pcs, np.cumsum(ratio) * 100, color='0.4', linestyle='--')
    cumulative.set_ylabel('Cumulative (%)', color='0.4')
    cumulative.spines['top'].set_visible(False)
    for ax in axes:
        ax.spines[['top', 'right']].set_visible(False)
        ax.tick_params(labelsize=7)
    fig.suptitle(f'HVG{n_hvg} | unintegrated PCA', fontsize=10)
    fig.subplots_adjust(left=0.07, right=0.93, top=0.84, bottom=0.34, wspace=0.52)
    buffer = BytesIO()
    fig.savefig(buffer, format='png', dpi=PCA_DISPLAY_DPI, facecolor='white')
    display(Image(data=buffer.getvalue()))
    plt.close(fig)


## Final CSS result

HVG2000, reference resolution 0.4, neighbors 10, final Leiden 0.8.
UMAP and DotPlot use the same 36 source-annotated categories.


In [ ]:
#11. =========== Source annotations and marker counts
MARKER_GROUPS = {'Otic origin': ['Foxg1', 'Six1', 'Eya1', 'Isl1', 'Pou4f1', 'Fgf10', 'Shox2'],
 'Early neurogenesis': ['Neurog1', 'Insm1', 'Neurod1', 'Sox2', 'Hes6', 'Gadd45g', 'Dll1'],
 'Cycling': ['Mki67', 'Top2a', 'Cdk1', 'Ccnb1', 'Mcm3', 'Cdt1', 'Ccna2', 'Tk1'],
 'Delamination / differentiation': ['Miat',
                                    'Tmsb4x',
                                    'St18',
                                    'Ebf1',
                                    'Zeb1',
                                    'Zeb2',
                                    'Myt1',
                                    'Dcx'],
 'SGN differentiation': ['Epha5', 'Shh', 'Meis2', 'Pou4f2', 'Prox1', 'Zfpm2'],
 'Early SGN programs': ['Nhlh1', 'Ebf2', 'Id1', 'Zfhx3', 'Mafb', 'Myt1l'],
 'Ia / II precursor': ['Tle4', 'Gata3', 'Gfra1', 'Islr2'],
 'Type Ia': ['Calb2', 'Pcdh20', 'Rxrg', 'Trim54', 'B3gat1', 'Cacna1b', 'Cpne9', 'Vsnl1'],
 'Type Ib': ['Calb1', 'Ttn', 'Kcnd2', 'Pls3', 'Plcb4', 'Kcnh2', 'Edil3'],
 'Type Ic': ['Lypd1', 'Grm8', 'Kcnc2', 'Sphkap', 'Cd164l2', 'Rgs12', 'Pcsk2', 'Mgat4c', 'Espn'],
 'Type II': ['Ngfr',
             'Plk5',
             'Th',
             'Calca',
             'Cux2',
             'Npy',
             'Cilp',
             'Grik3',
             'Prph',
             'Efna5',
             'Tac1',
             'Sox9',
             'Igfbpl1'],
 'Ib/Ic shared': ['Ntng1', 'Nalcn', 'Ncald', 'Runx1'],
 'Transition candidates': ['Tshz3', 'Rora'],
 'Apex-associated program': ['Etv4'],
 'Development': ['Cxcl14', 'Gabrg2'],
 'Pan-neuronal': ['Tubb3',
                  'Ncam1',
                  'Nefl',
                  'Nefm',
                  'Elavl3',
                  'Elavl4',
                  'Stmn2',
                  'Map2',
                  'Gap43',
                  'Uchl1',
                  'Actl6b',
                  'Snap25',
                  'Rbfox3',
                  'Nefh'],
 'Additional synaptic function': ['Rab3a', 'Vamp2', 'Sv2a', 'Rims1', 'Syt1'],
 'Exclusion: VGN candidates': ['Tlx3', 'Tlx1', 'Onecut1', 'Onecut2', 'Sall3', 'Casz1'],
 'Exclusion: geniculate / visceral': ['Phox2b', 'Phox2a', 'Tlx2', 'Ret', 'Hand2'],
 'Exclusion: CNS candidates': ['Barhl2', 'Evx1', 'Lhx2', 'Hoxb2', 'Pax6'],
 'Exclusion: hair cells': ['Atoh1', 'Pou4f3', 'Gfi1', 'Myo7a', 'Otof', 'Lhfpl5'],
 'Exclusion: glia': ['Sox10', 'Plp1', 'Mpz', 'S100b', 'Fabp7', 'Mbp', 'Pmp22'],
 'Exclusion: mesenchyme': ['Prrx1', 'Col1a1', 'Col1a2', 'Col3a1', 'Dcn', 'Pdgfra', 'Twist1'],
 'Exclusion: epithelial': ['Epcam', 'Krt8', 'Krt18'],
 'Exclusion: immune': ['Ptprc',
                       'Lyz2',
                       'Tyrobp',
                       'Fcer1g',
                       'C1qa',
                       'C1qb',
                       'Spi1',
                       'Csf1r',
                       'Ctss'],
 'Exclusion: erythroid': ['Gypa', 'Klf1', 'Gata1', 'Alas2', 'Slc4a1', 'Hba-a1', 'Hbb-bs'],
 'Exclusion: vessel': ['Pecam1', 'Kdr']}

EMBRYONIC_SOURCE_ORDER = [
    'E9.5 | CVG neuroblast',
    'E10.5 | O-2: delaminating otic/CVG neuroblast',
    'E10.5 | O-3: developing CVG neuron (not SGN-specific)',
    'E11.5 | Undifferentiated / proliferative CVG',
    'E11.5 | Differentiating SGN',
    'E13.5 | Late CVG',
    'E13.5 | Differentiating SGN',
    'E14.5 | Type Ia + Type II precursor candidate',
    'E14.5 | Type Ib + Type Ic precursor candidate',
    'E14.5 | Undifferentiated SGN',
    'E15.5 | Type Ia + Type II precursor candidate',
    'E15.5 | Type Ib + Type Ic precursor candidate',
    'E15.5 | Undifferentiated SGN',
    'E16.5 | Emerging Type Ic',
    'E16.5 | Type Ia + Type II precursor candidate',
    'E16.5 | Type Ib + Type Ic precursor candidate',
    'E16.5 | Undifferentiated SGN',
    'E17.5 | Type Ib + Type Ic precursor candidate',
    'E17.5 | Type Ia + Type II precursor candidate',
    'E18.5 | Immature Type II SGN',
    'E18.5 | Immature Type Ia SGN',
    'E18.5 | Immature Type Ib SGN',
    'E18.5 | Immature Type Ic SGN',
    'E18.5 | Undifferentiated SGN',
]
EMBRYONIC_SOURCE_PALETTE = {
    'E9.5 | CVG neuroblast': '#b08c43',
    'E10.5 | O-2: delaminating otic/CVG neuroblast': '#e08bbd',
    'E10.5 | O-3: developing CVG neuron (not SGN-specific)': '#5578e0',
    'E11.5 | Undifferentiated / proliferative CVG': '#438cb0',
    'E11.5 | Differentiating SGN': '#e08f55',
    'E13.5 | Late CVG': '#e05578',
    'E13.5 | Differentiating SGN': '#4dc9c9',
    'E14.5 | Type Ia + Type II precursor candidate': '#b06d78',
    'E14.5 | Type Ib + Type Ic precursor candidate': '#6db08f',
    'E14.5 | Undifferentiated SGN': '#79b043',
    'E15.5 | Type Ia + Type II precursor candidate': '#8ac97d',
    'E15.5 | Type Ib + Type Ic precursor candidate': '#b0946d',
    'E15.5 | Undifferentiated SGN': '#d98be0',
    'E16.5 | Emerging Type Ic': '#e06c55',
    'E16.5 | Type Ia + Type II precursor candidate': '#c94d95',
    'E16.5 | Type Ib + Type Ic precursor candidate': '#8b92e0',
    'E16.5 | Undifferentiated SGN': '#e0b255',
    'E17.5 | Type Ib + Type Ic precursor candidate': '#b0b043',
    'E17.5 | Type Ia + Type II precursor candidate': '#e0998b',
    'E18.5 | Immature Type II SGN': '#b06743',
    'E18.5 | Immature Type Ia SGN': '#9ab06d',
    'E18.5 | Immature Type Ib SGN': '#9f65c9',
    'E18.5 | Immature Type Ic SGN': '#4dc98b',
    'E18.5 | Undifferentiated SGN': '#55bee0',
}

symbols = np.array([g for genes in MARKER_GROUPS.values() for g in genes])
marker_groups = np.array([group for group, genes in MARKER_GROUPS.items() for _ in genes])
assert len(set(symbols)) == len(symbols)
source_labels = pd.Series(index=a.obs_names, dtype=object)
raw_blocks = []
for stage, part in parts.items():
    raw = np.full((part.n_obs, len(symbols)), np.nan)
    archived_symbols = pd.Index(part.uns['marker_gene_symbols'].astype(str)) if stage.startswith('E') else pd.Index([])
    if stage.startswith('E'):
        source_labels.loc[part.obs_names] = part.obs.stage_celltype.astype(str)
        archived = np.asarray(part.obsm['marker_raw_counts'], dtype=np.float64)
        measured = np.asarray(part.obsm['marker_measured'], dtype=bool)
        assert np.array_equal(np.isfinite(archived), measured), stage
    else:
        source_labels.loc[part.obs_names] = part.obs.input_stage.astype(str) + ' | ' + part.obs.cell_type.astype(str)
    source_symbols = part.var.gene_symbol.astype(str).to_numpy()
    for j, gene in enumerate(symbols):
        if gene in archived_symbols:
            raw[:, j] = archived[:, archived_symbols.get_loc(gene)]
        else:
            columns = np.flatnonzero(source_symbols == gene)
            if len(columns):
                raw[:, j] = np.asarray(part.layers[COUNTS_LAYER][:, columns].sum(axis=1)).ravel()
    raw_blocks.append(raw)

source_order = EMBRYONIC_SOURCE_ORDER + [
    stage + ' | ' + identity for stage in ('P1', 'P7', 'P28')
    for identity in ('Type Ia', 'Type Ib', 'Type Ic', 'Type II')
]
assert source_labels.notna().all() and set(source_labels) == set(source_order)
a.obs['source_stage_celltype'] = pd.Categorical(source_labels, categories=source_order)
marker_raw = np.vstack(raw_blocks)
marker_measured = np.isfinite(marker_raw)
assert marker_measured.any(axis=0).all()
denominator = np.asarray(a.layers[COUNTS_LAYER].sum(axis=1)).ravel().astype(np.float64)
marker_log = np.log1p(marker_raw * TARGET_SUM / denominator[:, None])
marker_data = dict(gene_symbols=symbols, marker_groups=marker_groups,
                   raw=marker_raw, measured=marker_measured, log=marker_log)


In [ ]:
#12. =========== CSS reference functions
from scipy.stats import rankdata
from sklearn.preprocessing import scale as sklearn_scale

def _css_array_hash(values):
    values = np.ascontiguousarray(values)
    digest = hashlib.sha256()
    digest.update(str(values.shape).encode())
    digest.update(str(values.dtype).encode())
    digest.update(values.tobytes())
    return digest.hexdigest()


def _css_cor_dense(x, y):
    """Official corSparse dense arithmetic, applied to average-tied ranks."""
    n = x.shape[0]
    mu_x = np.ravel(x.mean(0))
    mu_y = np.ravel(y.mean(0))
    covariance = (x.T.dot(y) - n * mu_x[:, None].dot(mu_y[:, None].T)) / (n - 1)
    sd_x = np.ravel(np.sqrt(((x ** 2).sum(0) - n * mu_x ** 2) / (n - 1)))
    sd_y = np.ravel(np.sqrt(((y ** 2).sum(0) - n * mu_y ** 2) / (n - 1)))
    return np.array(covariance / sd_x[:, None].dot(sd_y[:, None].T))


def compute_css_representation(a, ordered_genes, matched_pca, parameters, *, n_threads):
    """Return CSS and real batch-reference evidence; never normalize or relabel.

    Required parameters: batch_key, expected_batch_count, reference_pcs,
    reference_neighbors, reference_resolution, min_reference_clusters,
    random_state, leiden_flavor, leiden_directed, leiden_iterations.
    Dense expression is essential: zeros participate in average tied ranks.
    """
    p = dict(parameters)
    genes = pd.Index(ordered_genes).astype(str)
    pc = np.asarray(matched_pca)
    assert genes.is_unique and genes.isin(a.var_names).all()
    assert pc.shape == (a.n_obs, p['reference_pcs']) and np.isfinite(pc).all()
    assert a.obs[p['batch_key']].notna().all()
    assert a.obs[p['batch_key']].nunique() == p['expected_batch_count']
    expression = a[:, genes].X
    dense = expression.toarray() if sparse.issparse(expression) else np.asarray(expression).copy()
    dense = dense.astype(np.float64)
    assert np.isfinite(dense).all() and dense.min() >= 0
    work = ad.AnnData(X=dense, obs=a.obs[[p['batch_key']]].copy(),
                      var=pd.DataFrame(index=genes))
    work.obsm['X_pca'] = pc.copy()
    batch = work.obs[p['batch_key']].astype(str).to_numpy()
    labels = np.empty(a.n_obs, dtype=object)
    reference_objects, evidence, batch_summary = [], [], []
    with threadpool_limits(limits=n_threads):
        for name in pd.unique(batch):
            ix = np.flatnonzero(batch == name)
            local = ad.AnnData(X=work.X[ix],
                obs=pd.DataFrame(index=work.obs_names[ix]), var=work.var.copy())
            local.obsm['X_pca'] = work.obsm['X_pca'][ix]
            effective_k = min(p['reference_neighbors'], len(ix) - 1)
            assert effective_k >= 2, 'A reference batch is too small to cluster.'
            sc.pp.neighbors(local, use_rep='X_pca', n_neighbors=effective_k,
                            metric='euclidean', random_state=p['random_state'])
            sc.tl.leiden(local, resolution=p['reference_resolution'],
                flavor=p['leiden_flavor'], directed=p['leiden_directed'],
                n_iterations=p['leiden_iterations'], random_state=p['random_state'],
                key_added='reference_cluster')
            cluster = local.obs.reference_cluster.astype(str).to_numpy()
            levels = sorted(pd.unique(cluster), key=int)
            labels[ix] = [name + '|' + c for c in cluster]
            include = len(levels) >= p['min_reference_clusters']
            batch_summary.append(dict(batch=name, n_cells=len(ix), n_clusters=len(levels),
                reference_included=include, effective_neighbors=effective_k))
            if include:
                means = np.vstack([local.X[cluster == c].mean(axis=0) for c in levels])
                names = [name + '|' + c for c in levels]
                reference_objects.append(ad.AnnData(X=means, var=work.var.copy(),
                                                   obs=pd.DataFrame(index=names)))
                evidence.append(dict(batch=name, clusters=names, mean_expression=means.copy()))
        assert reference_objects, 'No batch supplies enough real reference clusters.'
        # Match the original library by aligning sorted intersecting gene IDs.
        aligned = np.intersect1d(work.var_names, reference_objects[0].var_names)
        assert len(aligned) == len(genes)
        ranked_cells = rankdata(work[:, aligned].X.T, method='average', axis=0)
        blocks, undefined = [], []
        for entry, reference in zip(evidence, reference_objects):
            ranked_reference = rankdata(reference[:, aligned].X.T, method='average', axis=0)
            correlation = _css_cor_dense(ranked_cells, ranked_reference)
            invalid = (~np.isfinite(correlation).all(axis=1) |
                       (correlation.std(axis=1, ddof=0) <= 1e-12))
            undefined.append(dict(batch=entry['batch'], undefined_rows=int(invalid.sum())))
            assert not invalid.any(), 'Undefined correlation or batch-wise z scaling.'
            # Use ddof=0, as in the original Python implementation; R uses ddof=1.
            blocks.append(sklearn_scale(correlation, axis=1))
        css = np.concatenate(blocks, axis=1)
    assert css.shape[0] == a.n_obs and np.isfinite(css).all()
    history = dict(method='CSS', parameters=p, n_cells=a.n_obs, n_genes=len(genes),
        n_dimensions=css.shape[1], reference_batches=batch_summary,
        undefined_reference_rows=undefined, dense_average_tied_spearman=True,
        z_ddof=0, css_pca=False, n_threads=n_threads,
        reference_profile_sha256=[_css_array_hash(e['mean_expression']) for e in evidence],
        css_sha256=_css_array_hash(css),
        source_commit='9a81488f4cddbdc1689a8505231a145eab9c3a94',
        caveat='Batches with too few real clusters contribute no reference block; '
               'their cells are retained. No age, identity or region enters CSS.')
    return dict(embedding=css, cell_ids=np.asarray(a.obs_names).astype(str),
                ordered_genes=np.asarray(genes).astype(str), reference_labels=labels,
                reference_profiles=evidence, history=history)


def transform_css_reference(a, model):
    """Apply fitted CSS reference means to the same normalized gene axis."""
    assert np.array_equal(a.var_names.astype(str), model['common_genes'])
    assert a.uns['preprocessing']['target_sum'] == model['preprocessing']['target_sum']
    genes = pd.Index(model['ordered_genes'])
    aligned = np.intersect1d(a.var_names, genes)
    assert len(aligned) == len(genes)
    expression = a[:, aligned].X
    dense = expression.toarray() if sparse.issparse(expression) else np.asarray(expression)
    assert np.isfinite(dense).all() and dense.min() >= 0
    order = genes.get_indexer(aligned)
    blocks = []
    with threadpool_limits(limits=N_THREADS):
        ranked = rankdata(dense.astype(np.float64).T, method='average', axis=0)
        for entry in model['reference_profiles']:
            means = entry['mean_expression'][:, order]
            reference = rankdata(means.T, method='average', axis=0)
            correlation = _css_cor_dense(ranked, reference)
            assert np.isfinite(correlation).all()
            assert (correlation.std(axis=1, ddof=0) > 1e-12).all()
            blocks.append(sklearn_scale(correlation, axis=1))
    return np.concatenate(blocks, axis=1)




In [ ]:
#13. =========== Selected CSS reference
SELECTED_HVG = 2000
REFERENCE_RESOLUTION = 0.4
REFERENCE_NEIGHBORS = 15
MIN_REFERENCE_CLUSTERS = 3

# Verify the selected run; revalidate after upstream changes.
EXPECTED_PACKAGES = {'scanpy': '1.11.5', 'anndata': '0.12.10', 'numpy': '2.4.3', 'scipy': '1.17.1', 'pandas': '2.3.3', 'scikit-learn': '1.8.0', 'igraph': '1.0.0', 'leidenalg': '0.11.0', 'umap-learn': '0.5.11', 'pynndescent': '0.6.0', 'matplotlib': '3.10.8'}
EXPECTED_COUNTS_SHA256 = '1a07e73236b2e308d35cde791b504fb1fd48e3e5c883785a78b094c8386fc66d'
EXPECTED_PCA_SHA256 = 'a0a4b919978fc9b050092b8b52634a487d2f96f2d4af0b5dfcd2197275640982'
EXPECTED_CSS_SHA256 = '5060c51b82a069b8d2770a8580466ab7d347ef15b2e7db28d47740cc7103ea45'

CSS_PARAMETERS = dict(
    batch_key=BATCH_KEY, expected_batch_count=a.obs[BATCH_KEY].nunique(),
    reference_pcs=N_PCS, reference_neighbors=REFERENCE_NEIGHBORS,
    reference_resolution=REFERENCE_RESOLUTION,
    min_reference_clusters=MIN_REFERENCE_CLUSTERS, random_state=RANDOM_SEED,
    leiden_flavor='igraph', leiden_directed=False, leiden_iterations=2,
)

def resolution_code(value):
    return format(float(value), '.12g').replace('.', 'p')

assert a.uns['preparation_complete']
assert package_versions == EXPECTED_PACKAGES, 'Use the validated scvi_env package versions.'
assert joint_counts_hash == EXPECTED_COUNTS_SHA256
assert _css_array_hash(branches[SELECTED_HVG].obsm['X_matched_pca']) == EXPECTED_PCA_SHA256
fit_id = f'h{resolution_code(SELECTED_HVG / 1000)}_r{resolution_code(REFERENCE_RESOLUTION)}'
fit = compute_css_representation(
    a, hvg_genes[SELECTED_HVG], branches[SELECTED_HVG].obsm['X_matched_pca'],
    CSS_PARAMETERS, n_threads=N_THREADS
)
assert np.array_equal(fit['cell_ids'], a.obs_names.astype(str))
assert fit['history']['css_sha256'] == EXPECTED_CSS_SHA256, 'CSS differs from the accepted grid.'


In [ ]:
#14. =========== CSS graph, UMAP and final Leiden
N_NEIGHBORS = 10
FINAL_LEIDEN_RESOLUTION = 0.8
UMAP_MIN_DIST = 0.35
UMAP_SPREAD = 1.0
UMAP_EPOCHS = 1000
EXPECTED_UMAP_SHA256 = '4cd0a00aaccc010c8f8ed2f258462ab1524d2df93f9541396001c9bda7f2254b'
EXPECTED_CLUSTER_SHA256 = '202dad0cbebbd1024bde96a75aba7a94732f1b4d11d057258b6cbc6fd663743f'

UMAP_PARAMETERS = dict(min_dist=UMAP_MIN_DIST, spread=UMAP_SPREAD, maxiter=UMAP_EPOCHS,
                       random_state=RANDOM_SEED, init_pos='spectral', alpha=1.0,
                       gamma=1.0, negative_sample_rate=5)
graph_id = f'{fit_id}_k{N_NEIGHBORS}'
final_run = f'{graph_id}_l{resolution_code(FINAL_LEIDEN_RESOLUTION)}'
graph = ad.AnnData(X=np.empty((a.n_obs, 0), dtype=np.float32),
                  obs=a.obs[[STUDY_KEY, BATCH_KEY, 'input_stage', 'source_stage_celltype']].copy())
graph.obsm['X_css'] = fit['embedding']
with threadpool_limits(limits=N_THREADS):
    sc.pp.neighbors(graph, use_rep='X_css', n_pcs=None, n_neighbors=N_NEIGHBORS,
                    metric='euclidean', random_state=RANDOM_SEED)
    sc.tl.umap(graph, **UMAP_PARAMETERS)
    sc.tl.leiden(graph, resolution=FINAL_LEIDEN_RESOLUTION, flavor='igraph', directed=False,
                 n_iterations=2, random_state=RANDOM_SEED, key_added='leiden_css')
xy = graph.obsm['X_umap'].astype(np.float32).copy()
assert _css_array_hash(xy) == EXPECTED_UMAP_SHA256
assert _css_array_hash(np.asarray(graph.obs.leiden_css.astype(str), dtype=str)) == EXPECTED_CLUSTER_SHA256


In [ ]:
#15. =========== Figure functions
import textwrap
import matplotlib.patheffects as pe
from matplotlib.colors import LinearSegmentedColormap, Normalize
from matplotlib.transforms import Bbox
from scipy.spatial.distance import cdist

def _medoid(points):
    """An actual cell minimizing summed Euclidean distances; stable ties."""
    best_score, best_index = np.inf, 0
    for start in range(0, len(points), 256):
        scores = cdist(points[start:start + 256], points).sum(axis=1)
        local = int(np.argmin(scores))
        if scores[local] < best_score:
            best_score, best_index = scores[local], start + local
    return points[best_index]


def _label_text(value, display_types, width):
    value = str(display_types.get(value, value) if display_types else value)
    value = value.replace('_', ' ')
    if ' | ' in value:
        stage, name = value.split(' | ', 1)
        return stage + '\n' + textwrap.fill(name, width=width, break_long_words=False)
    return textwrap.fill(value, width=width, break_long_words=False)


def _place_labels(fig, ax, coords, labels, levels, texts, fontsize):
    anchors = [_medoid(coords[labels == level]) for level in levels]
    artists = [ax.text(*anchor, text, ha='center', va='center', fontsize=fontsize,
                       color='black', linespacing=1.1, zorder=5,
                       path_effects=[pe.withStroke(linewidth=1.45, foreground='white')])
               for anchor, text in zip(anchors, texts)]
    fig.canvas.draw()
    renderer = fig.canvas.get_renderer()
    bounds = ax.get_window_extent(renderer).padded(-3)
    anchors_px = ax.transData.transform(np.asarray(anchors))
    sizes = [(t.get_window_extent(renderer).width + 4,
              t.get_window_extent(renderer).height + 4) for t in artists]
    occupied, placements = [], {}
    # Make room for long labels while preserving their preferred anchors.
    order = sorted(range(len(artists)), key=lambda i: -sizes[i][0] * sizes[i][1])
    radii = np.arange(0, 0.8 * max(bounds.width, bounds.height), 10.0)
    for i in order:
        width, height = sizes[i]
        anchor = anchors_px[i]
        best = None
        for radius in radii:
            angles = np.linspace(0, 2 * np.pi, max(1, int(2 * np.pi * radius / 12)),
                                 endpoint=False)
            for angle in angles:
                center = anchor + radius * np.array([np.cos(angle), np.sin(angle)])
                center[0] = np.clip(center[0], bounds.x0 + width / 2, bounds.x1 - width / 2)
                center[1] = np.clip(center[1], bounds.y0 + height / 2, bounds.y1 - height / 2)
                box = Bbox.from_bounds(center[0] - width / 2, center[1] - height / 2,
                                       width, height)
                if any(box.overlaps(other) for other in occupied):
                    continue
                distance = float(np.sum((center - anchor) ** 2))
                if best is None or distance < best[0]:
                    best = (distance, center, box)
            if best is not None:
                break
        if best is None:
            raise ValueError('UMAP labels do not fit; increase axis_size or reduce label_font.')
        placements[i] = best[1]
        occupied.append(best[2])
        artists[i].set_position(ax.transData.inverted().transform(best[1]))
    # Connect only displaced labels; shrink the line to the label boundary.
    for i, artist in enumerate(artists):
        delta = anchors_px[i] - placements[i]
        boundary = np.max(np.abs(delta) / (np.asarray(sizes[i]) / 2))
        if np.linalg.norm(delta) > 12 and boundary > 1:
            start = placements[i] + delta / boundary
            line = ax.transData.inverted().transform([start, anchors_px[i]])
            ax.plot(line[:, 0], line[:, 1], color='0.32', lw=0.42, alpha=0.70, zorder=3)
    fig.canvas.draw()
    renderer = fig.canvas.get_renderer()
    boxes = [artist.get_window_extent(renderer) for artist in artists]
    assert not any(left.overlaps(right) for i, left in enumerate(boxes)
                   for right in boxes[i + 1:]), 'UMAP labels overlap.'
    return artists


def save_umap_pair(coords, labels, category_order, palette, title, out_path, dpi,
                   point_size=1.35, point_alpha=0.82, label_font=6.75,
                   legend_font=6.8, axis_size=5.6, display_types=None, random_seed=0):
    """Save separate legend (_u) and label (_ul) PNGs without displaying them."""
    coords = np.asarray(coords, dtype=float)
    labels = np.asarray(labels, dtype=str)
    assert coords.shape == (len(labels), 2) and np.isfinite(coords).all()
    levels = [str(level) for level in category_order if np.any(labels == str(level))]
    assert set(levels) == set(labels) and len(levels) == len(set(levels))
    assert set(levels).issubset(palette)
    center = (coords.min(axis=0) + coords.max(axis=0)) / 2
    span = max(float(np.ptp(coords, axis=0).max()) * 1.12, 1.0)
    limits = [(value - span / 2, value + span / 2) for value in center]
    point_order = np.random.default_rng(random_seed).permutation(len(labels))
    colors = np.asarray([palette[label] for label in labels])
    counts = {level: int(np.sum(labels == level)) for level in levels}
    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    paths = []
    for mode, suffix in [('legend', '_u'), ('label', '_ul')]:
        left, bottom, top = 0.60, 0.52, 0.52
        extra = (4.75 if len(levels) > 20 else 2.65) if mode == 'legend' else 0.22
        width, height = left + axis_size + extra, bottom + axis_size + top
        fig = plt.figure(figsize=(width, height), dpi=100, facecolor='white')
        try:
            ax = fig.add_axes([left / width, bottom / height, axis_size / width,
                               axis_size / height])
            ax.set_facecolor('white')
            ax.grid(False)
            ax.scatter(coords[point_order, 0], coords[point_order, 1],
                       c=colors[point_order], s=point_size, alpha=point_alpha,
                       linewidths=0, edgecolors='none')
            ax.set(xlim=limits[0], ylim=limits[1], xlabel='UMAP1', ylabel='UMAP2')
            ax.set_box_aspect(1)
            ax.set_title(title, fontsize=9, pad=11)
            ax.tick_params(labelsize=7, width=0.8, length=3)
            for side in ('top', 'right'):
                ax.spines[side].set_visible(False)
            for side in ('left', 'bottom'):
                ax.spines[side].set_color('black')
                ax.spines[side].set_linewidth(1.1)
            if mode == 'legend':
                handles = [Line2D([], [], marker='o', ls='', markersize=5,
                                  markerfacecolor=palette[level], markeredgecolor='none')
                           for level in levels]
                legend_labels = [_label_text(level, display_types, 29)
                                 + f' (n={counts[level]:,})' for level in levels]
                legend = ax.legend(handles, legend_labels, loc='center left',
                                   bbox_to_anchor=(1.025, 0.5), frameon=False,
                                   fontsize=legend_font, ncol=2 if len(levels) > 20 else 1,
                                   handletextpad=0.35, labelspacing=0.48,
                                   columnspacing=0.9, borderaxespad=0)
                fig.canvas.draw()
                extent = legend.get_window_extent(fig.canvas.get_renderer())
                if extent.x1 > fig.bbox.x1 - 12:
                    width += (extent.x1 - fig.bbox.x1 + 14) / fig.dpi
                    fig.set_size_inches(width, height)
                    ax.set_position([left / width, bottom / height,
                                     axis_size / width, axis_size / height])
                fig.canvas.draw()
                extent = legend.get_window_extent(fig.canvas.get_renderer())
                assert extent.y0 >= 5 and extent.y1 <= fig.bbox.y1 - 5, 'Legend exceeds height.'
            else:
                texts = [_label_text(level, display_types, 27) for level in levels]
                _place_labels(fig, ax, coords, labels, levels, texts, label_font)
            fig.canvas.draw()
            box = ax.get_window_extent(fig.canvas.get_renderer())
            assert abs(box.width - box.height) < 0.5, 'UMAP axes are not square.'
            path = out_path.with_name(out_path.stem + suffix + '.png')
            fig.savefig(path, dpi=dpi, facecolor='white')
            paths.append(path)
        finally:
            plt.close(fig)
    return tuple(paths)


def save_dotplot(stats, marker_data, scaled, title, path, dpi, max_dot_area=36,
                 column_gap_ratio=0.5, row_pitch=0.17, gene_font=6.3, row_font=6.5,
                 group_font=6.5, group_rotation=90):
    """Plot supplied annotated-type statistics and global gene z-scores."""
    genes = np.asarray(marker_data['gene_symbols'], dtype=str)
    groups = np.asarray(marker_data['marker_groups'], dtype=str)
    levels = np.asarray(stats['levels'], dtype=str)
    pct = np.asarray(stats['pct'], dtype=float)
    coverage = np.asarray(stats['coverage'], dtype=float)
    scaled = np.asarray(scaled, dtype=float)
    n_cells = np.asarray(stats['n_cells'], dtype=int)
    n_rows, n_genes = len(levels), len(genes)
    assert len(set(genes)) == n_genes and len(groups) == n_genes
    assert pct.shape == coverage.shape == scaled.shape == (n_rows, n_genes)
    assert n_cells.shape == (n_rows,) and (n_cells > 0).all()
    assert np.isfinite(coverage).all() and ((coverage >= 0) & (coverage <= 1)).all()
    measured = coverage > 0
    assert np.isfinite(pct[measured]).all() and np.isfinite(scaled[measured]).all()
    assert ((pct[measured] >= 0) & (pct[measured] <= 100)).all()
    assert max_dot_area > 0 and row_pitch > 0 and column_gap_ratio >= 0
    # Matplotlib's circle marker has rendered diameter sqrt(s) points.
    largest = float(np.max(pct[measured], initial=0)) / 100
    diameter_pt = np.sqrt(max_dot_area * largest) if largest else np.sqrt(max_dot_area)
    pitch = (1 + column_gap_ratio) * diameter_pt / 72
    matrix_width = n_genes * pitch
    matrix_height = n_rows * max(row_pitch, diameter_pt / 72 * 1.15)
    left, right, bottom, top = 1.08, 0.16, 0.77, 1.26
    width, height = left + matrix_width + right, bottom + matrix_height + top
    fig = plt.figure(figsize=(width, height), dpi=100, facecolor='white')
    try:
        ax = fig.add_axes([left / width, bottom / height,
                           matrix_width / width, matrix_height / height])
        ax.set_facecolor('white')
        xx, yy = np.meshgrid(np.arange(n_genes), np.arange(n_rows))
        cmap = LinearSegmentedColormap.from_list(
            'marker_purple', ['#F1F1F1', '#E8C6DE', '#C67ABA', '#952E7D', '#69005F'])
        norm = Normalize(-1, 1, clip=True)
        positive = measured & (pct > 0)
        points = ax.scatter(xx[positive], yy[positive],
                            s=max_dot_area * pct[positive] / 100,
                            c=scaled[positive], cmap=cmap, norm=norm,
                            edgecolors='none', linewidths=0)
        missing = ~measured
        if missing.any():
            ax.scatter(xx[missing], yy[missing], marker='x', s=max_dot_area * 0.30,
                       c='0.67', linewidths=0.4)
        partial = np.any(coverage < 1, axis=0)
        ax.set_xticks(np.arange(n_genes),
                      [gene + ('\u2020' if flag else '') for gene, flag in zip(genes, partial)],
                      rotation=55, ha='right', fontsize=gene_font)
        row_labels = [level.replace('_', ' ') + f' (n={n:,})' for level, n in zip(levels, n_cells)]
        ax.set_yticks(np.arange(n_rows), row_labels, fontsize=row_font)
        ax.set(xlim=(-0.5, n_genes - 0.5), ylim=(n_rows - 0.5, -0.5))
        ax.tick_params(length=0, pad=2)
        ax.grid(False)
        for spine in ax.spines.values():
            spine.set_visible(False)
        starts = np.r_[0, np.flatnonzero(groups[1:] != groups[:-1]) + 1]
        ends = np.r_[starts[1:], n_genes]
        assert len(set(groups)) == len(starts), 'Marker groups must be contiguous.'
        group_artists = []
        for i, (start, end) in enumerate(zip(starts, ends)):
            x0, x1 = (start + 0.12) / n_genes, (end - 0.12) / n_genes
            ax.plot([x0, x1], [1.025, 1.025], transform=ax.transAxes,
                    clip_on=False, color=MARKER_COLORS[groups[start]], lw=2)
            heading = groups[start]
            group_artists.append(ax.text((x0 + x1) / 2, 1.065, heading,
                                transform=ax.transAxes, ha='center', va='bottom',
                                rotation=group_rotation, fontsize=group_font, clip_on=False))
        fig.canvas.draw()
        renderer = fig.canvas.get_renderer()
        label_width = max(text.get_window_extent(renderer).width for text in ax.get_yticklabels()) / fig.dpi
        left = max(left, label_width + 0.12)
        width = left + matrix_width + right
        heading_height = (max(text.get_window_extent(renderer).y1 for text in group_artists)
                          - ax.get_window_extent(renderer).y1) / fig.dpi
        top = max(top, heading_height + 0.83)
        height = bottom + matrix_height + top
        fig.set_size_inches(width, height)
        ax.set_position([left / width, bottom / height,
                         matrix_width / width, matrix_height / height])
        fig.text(left / width, (height - 0.18) / height, title,
                 ha='left', va='top', fontsize=8)
        legend_y = (bottom + matrix_height + heading_height + 0.25) / height
        cax = fig.add_axes([left / width, legend_y, matrix_width * 0.40 / width, 0.07 / height])
        colorbar = fig.colorbar(points, cax=cax, orientation='horizontal',
                               ticks=[-1, -0.5, 0, 0.5, 1])
        colorbar.outline.set_visible(False)
        colorbar.ax.tick_params(labelsize=5.8, length=2, pad=1)
        cax.set_title('Avg. Exp. (gene-wise z)', fontsize=6.5, loc='left', pad=4)
        handles = [Line2D([], [], marker='o', ls='', markerfacecolor='0.5',
                          markeredgecolor='none', markersize=np.sqrt(max_dot_area * value / 100))
                   for value in (0, 25, 50, 75, 100)]
        fig.legend(handles, ['0', '25', '50', '75', '100'], title='% Exp.',
                   loc='lower right', bbox_to_anchor=((left + matrix_width) / width,
                                                     legend_y - 0.03 / height),
                   frameon=False, ncol=5, fontsize=6, title_fontsize=6.5,
                   handlelength=0.7, handletextpad=0.3, columnspacing=0.7, borderaxespad=0)
        fig.text(left / width, 0.045 / height,
                 '\u2020 Not measured in all sources; % among measured cells.  \u00d7 No measured cells.',
                 fontsize=5.6, ha='left', va='bottom')
        fig.canvas.draw()
        renderer = fig.canvas.get_renderer()
        pixel_pitch = np.diff(ax.transData.transform([[0, 0], [1, 0]])[:, 0])[0]
        pixel_diameter = (points.get_paths()[0].get_extents().width
                          * points.get_transforms()[:, 0, 0].max()
                          if positive.any() else diameter_pt * fig.dpi / 72)
        assert np.isclose(pixel_pitch / pixel_diameter, 1 + column_gap_ratio, atol=0.025)
        text_boxes = [text.get_window_extent(renderer) for text in [*ax.get_xticklabels(), *ax.get_yticklabels()]]
        assert all(box.x0 >= 0 and box.y0 > 0 and box.x1 <= fig.bbox.width and box.y1 <= fig.bbox.height for box in text_boxes), 'Labels are clipped.'
        group_boxes = [text.get_window_extent(renderer) for text in group_artists]
        assert not any(a.overlaps(b) for i, a in enumerate(group_boxes)
                       for b in group_boxes[i + 1:]), 'Marker group headings overlap.'
        path = Path(path).with_suffix('.png')
        path.parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(path, dpi=dpi, facecolor='white')
    finally:
        plt.close(fig)
    return path


In [ ]:
#16. =========== Annotated UMAP and full/split DotPlots
# Set the output directory path.
FIGURE_DIR = ROOT
FIGURE_DPI = 600
UMAP_POINT_SIZE = 1.35
UMAP_ALPHA = 0.82
UMAP_AXIS_SIZE = 5.6
UMAP_LABEL_FONT = 6.75
UMAP_LEGEND_FONT = 6.8
DOT_MAX_AREA = 36
DOT_COLUMN_GAP = 0.5
DOT_ROW_PITCH = 0.17
DOT_GENE_FONT = 6.3
DOT_TYPE_FONT = 6.5
DOT_GROUP_FONT = 6.5
DOT_GROUP_ROTATION = 90
POSTNATAL_SOURCE_PALETTE = {'P7 | Type Ia': '#55a5c3',
 'P1 | Type Ia': '#628ac8',
 'P7 | Type Ib': '#c3556e',
 'P28 | Type Ic': '#70cdb0',
 'P1 | Type Ib': '#70c862',
 'P28 | Type Ib': '#5855c3',
 'P7 | Type Ic': '#c7cd70',
 'P28 | Type Ia': '#c86e62',
 'P7 | Type II': '#c39d55',
 'P1 | Type Ic': '#a5c355',
 'P28 | Type II': '#aa55c3',
 'P1 | Type II': '#70cdc7'}
DOT_ROW_ORDER = ['E9.5 | CVG neuroblast',
 'E10.5 | O-2: delaminating otic/CVG neuroblast',
 'E10.5 | O-3: developing CVG neuron (not SGN-specific)',
 'E11.5 | Undifferentiated / proliferative CVG',
 'E13.5 | Late CVG',
 'E11.5 | Differentiating SGN',
 'E13.5 | Differentiating SGN',
 'E14.5 | Undifferentiated SGN',
 'E15.5 | Undifferentiated SGN',
 'E16.5 | Undifferentiated SGN',
 'E18.5 | Undifferentiated SGN',
 'E14.5 | Type Ia + Type II precursor candidate',
 'E15.5 | Type Ia + Type II precursor candidate',
 'E16.5 | Type Ia + Type II precursor candidate',
 'E17.5 | Type Ia + Type II precursor candidate',
 'E14.5 | Type Ib + Type Ic precursor candidate',
 'E15.5 | Type Ib + Type Ic precursor candidate',
 'E16.5 | Type Ib + Type Ic precursor candidate',
 'E17.5 | Type Ib + Type Ic precursor candidate',
 'E18.5 | Immature Type Ia SGN',
 'P1 | Type Ia',
 'P7 | Type Ia',
 'P28 | Type Ia',
 'E18.5 | Immature Type Ib SGN',
 'P1 | Type Ib',
 'P7 | Type Ib',
 'P28 | Type Ib',
 'E16.5 | Emerging Type Ic',
 'E18.5 | Immature Type Ic SGN',
 'P1 | Type Ic',
 'P7 | Type Ic',
 'P28 | Type Ic',
 'E18.5 | Immature Type II SGN',
 'P1 | Type II',
 'P7 | Type II',
 'P28 | Type II']
DOT_PANELS = [('dev',
  ('Otic origin',
   'Early neurogenesis',
   'Cycling',
   'Delamination / differentiation',
   'SGN differentiation',
   'Early SGN programs',
   'Ia / II precursor')),
 ('sgn',
  ('Type Ia',
   'Type Ib',
   'Type Ic',
   'Type II',
   'Ib/Ic shared',
   'Transition candidates',
   'Apex-associated program',
   'Development',
   'Pan-neuronal',
   'Additional synaptic function')),
 ('exclude',
  ('Exclusion: VGN candidates',
   'Exclusion: geniculate / visceral',
   'Exclusion: CNS candidates',
   'Exclusion: hair cells',
   'Exclusion: glia',
   'Exclusion: mesenchyme',
   'Exclusion: epithelial',
   'Exclusion: immune',
   'Exclusion: erythroid',
   'Exclusion: vessel'))]
MARKER_COLORS = {'Otic origin': '#5B7F9D',
 'Early neurogenesis': '#8B759C',
 'Cycling': '#B29A59',
 'Delamination / differentiation': '#5C9084',
 'SGN differentiation': '#738D57',
 'Early SGN programs': '#9A6E9C',
 'Ia / II precursor': '#929E50',
 'Type Ia': '#B7864A',
 'Type Ib': '#B65F64',
 'Type Ic': '#557B9F',
 'Type II': '#519A94',
 'Ib/Ic shared': '#5C9084',
 'Transition candidates': '#738D57',
 'Apex-associated program': '#9A6E9C',
 'Development': '#929E50',
 'Pan-neuronal': '#6F748C',
 'Additional synaptic function': '#5B7F9D',
 'Exclusion: VGN candidates': '#8B759C',
 'Exclusion: geniculate / visceral': '#B29A59',
 'Exclusion: CNS candidates': '#5C9084',
 'Exclusion: hair cells': '#738D57',
 'Exclusion: glia': '#9A6E9C',
 'Exclusion: mesenchyme': '#929E50',
 'Exclusion: epithelial': '#6F748C',
 'Exclusion: immune': '#B22222',
 'Exclusion: erythroid': '#C0839E',
 'Exclusion: vessel': '#B29A59'}

source_palette = dict(EMBRYONIC_SOURCE_PALETTE, **POSTNATAL_SOURCE_PALETTE)
assert set(source_palette) == set(source_order) == set(DOT_ROW_ORDER)
labels = a.obs.source_stage_celltype.astype(str).to_numpy()
shape = (len(DOT_ROW_ORDER), len(symbols))
mean, pct = np.full(shape, np.nan), np.full(shape, np.nan)
n_cells, n_measured = np.zeros(shape[0], dtype=int), np.zeros(shape, dtype=int)
for i, label in enumerate(DOT_ROW_ORDER):
    keep = labels == label
    mask = marker_measured[keep]
    n_cells[i], n_measured[i] = keep.sum(), mask.sum(axis=0)
    np.divide(np.where(mask, marker_log[keep], 0).sum(axis=0), n_measured[i],
              out=mean[i], where=n_measured[i] > 0)
    np.divide(100 * ((marker_raw[keep] > 0) & mask).sum(axis=0), n_measured[i],
              out=pct[i], where=n_measured[i] > 0)
assert (n_cells > 0).all() and n_cells.sum() == a.n_obs
DOT_CENTER = np.nanmean(mean, axis=0)
DOT_SCALE = np.nanstd(mean, axis=0, ddof=0)
scaled = np.divide(mean - DOT_CENTER, DOT_SCALE, out=np.zeros_like(mean), where=DOT_SCALE > 0)
scaled[n_measured == 0] = np.nan
dot_stats = dict(levels=DOT_ROW_ORDER, n_cells=n_cells, mean=mean, pct=pct,
                 n_measured=n_measured, coverage=n_measured / n_cells[:, None])

FIGURE_DIR.mkdir(parents=True, exist_ok=True)
umap_style = dict(dpi=FIGURE_DPI, point_size=UMAP_POINT_SIZE, point_alpha=UMAP_ALPHA,
                  axis_size=UMAP_AXIS_SIZE, label_font=UMAP_LABEL_FONT,
                  legend_font=UMAP_LEGEND_FONT, random_seed=RANDOM_SEED)
# Set the output file path.
saved_figures = list(save_umap_pair(xy, labels, source_order, source_palette,
                                   graph_id + ' | source annotation',
                                   ROOT, **umap_style))
for suffix, groups in [('full', tuple(MARKER_GROUPS)), *DOT_PANELS]:
    columns = np.flatnonzero(np.isin(marker_groups, groups))
    panel_stats = dict(levels=DOT_ROW_ORDER, n_cells=n_cells,
                       pct=pct[:, columns], coverage=dot_stats['coverage'][:, columns])
    panel_markers = dict(gene_symbols=symbols[columns], marker_groups=marker_groups[columns])
    # Set the output file path.
    saved_figures.append(save_dotplot(
        panel_stats, panel_markers, scaled[:, columns], 'Whole-period CSS | annotated types',
        ROOT, dpi=FIGURE_DPI,
        max_dot_area=DOT_MAX_AREA, column_gap_ratio=DOT_COLUMN_GAP, row_pitch=DOT_ROW_PITCH,
        gene_font=DOT_GENE_FONT, row_font=DOT_TYPE_FONT, group_font=DOT_GROUP_FONT,
        group_rotation=DOT_GROUP_ROTATION
    ))
assert len(saved_figures) == len(set(saved_figures)) == 6
assert all(path.is_file() and path.stat().st_size > 0 for path in saved_figures)
assert not plt.get_fignums()


In [ ]:
#17. =========== Final CSS model and AnnData
import json
import joblib

# Set the stage input directory; uses integration H5ADs generated by the corresponding fig1_E*_clustering.ipynb notebook.
DATA_DIR = ROOT
# Set the result file path; uses Whole_CSS_model.joblib generated by whole-period CSS integration.
MODEL_PATH = ROOT
# Set the result file path; uses Whole_CSS.h5ad generated by whole-period CSS integration.
H5AD_PATH = ROOT

assert counts_sha256(a.layers[COUNTS_LAYER], a.obs_names, a.var_names) == joint_counts_hash
assert a.obs_names.equals(cell_order)
a.obsm['X_css'] = fit['embedding'].copy()
a.obsm['X_umap'] = xy.copy()
a.obsm['X_matched_pca'] = branches[SELECTED_HVG].obsm['X_matched_pca'].copy()
a.obsp['connectivities'] = graph.obsp['connectivities'].copy()
a.obsp['distances'] = graph.obsp['distances'].copy()
a.uns['neighbors'] = dict(graph.uns['neighbors'])
a.uns['umap'] = dict(graph.uns['umap'])
a.var['css_highly_variable'] = a.var_names.isin(fit['ordered_genes'])
a.obs['css_reference_cluster'] = pd.Categorical(fit['reference_labels'])
a.obs['leiden_css'] = graph.obs.leiden_css.copy()
a.uns['leiden_css'] = dict(resolution=FINAL_LEIDEN_RESOLUTION, random_state=RANDOM_SEED,
                           flavor='igraph', directed=False, n_iterations=2)
a.uns['source_stage_celltype_colors'] = np.array(
    [source_palette[level] for level in source_order], dtype=str)
a.obsm['css_marker_raw_counts'] = marker_data['raw'].copy()
a.obsm['css_marker_measured'] = marker_data['measured'].copy()
a.obsm['css_marker_log'] = marker_data['log'].copy()
a.uns['css_marker_gene_symbols'] = marker_data['gene_symbols'].copy()
a.uns['css_marker_groups'] = marker_data['marker_groups'].copy()
a.uns['css'] = dict(
    parameters=dict(CSS_PARAMETERS), ordered_genes=fit['ordered_genes'].copy(),
    history_json=json.dumps(fit['history']), model_file=MODEL_PATH.name,
    graph_id=graph_id, selected_run=final_run, selected_leiden_resolution=FINAL_LEIDEN_RESOLUTION,
    css_sha256=EXPECTED_CSS_SHA256, umap_sha256=EXPECTED_UMAP_SHA256,
    counts_sha256=joint_counts_hash,
)
a.uns['css_figure_contract'] = dict(
    source_order=np.array(source_order, dtype=str),
    source_colors=np.array([source_palette[x] for x in source_order], dtype=str),
    dot_center=DOT_CENTER.copy(), dot_scale=DOT_SCALE.copy(),
    calibration='Gene-wise z across 36 source-annotated types; ddof=0',
    dot_row_order=np.array(DOT_ROW_ORDER, dtype=str), dpi=FIGURE_DPI,
)

# Preserve missing QC flags and separate mixed text/numeric metadata.
mixed_field = 'qc_complexity_confidence'
text_field = mixed_field + '_source_text'
if mixed_field in a.obs and text_field not in a.obs:
    values = a.obs[mixed_field]
    text = values.map(lambda value: isinstance(value, str))
    a.obs[mixed_field] = pd.to_numeric(values.where(~text, np.nan), errors='raise').astype(float)
    a.obs[text_field] = pd.Categorical(values.where(text, None).astype(object))
    a.uns['obs_serialization'] = dict(
        mixed_column=mixed_field, text_column=text_field,
        restoration='Use source_text when present; otherwise use the numeric column.'
    )
for name in a.obs.select_dtypes(include=['object', 'string']).columns:
    values = a.obs[name]
    kind = pd.api.types.infer_dtype(values, skipna=True)
    if kind == 'boolean':
        a.obs[name] = values.astype('boolean')
    elif kind in ('string', 'empty'):
        a.obs[name] = pd.Categorical(values.astype(object))
    else:
        raise TypeError(f'Unsupported mixed metadata column: {name} ({kind})')

model = dict(
    format='whole-period-css-reference-v1', method='CSS', parameters=dict(CSS_PARAMETERS),
    common_genes=np.asarray(a.var_names, dtype=str), ordered_genes=fit['ordered_genes'].copy(),
    reference_profiles=fit['reference_profiles'], reference_labels=fit['reference_labels'].copy(),
    history=fit['history'], cell_ids=fit['cell_ids'].copy(),
    preprocessing=dict(a.uns['preprocessing']), counts_sha256=joint_counts_hash,
    input_manifest=a.uns['input_manifest'], package_versions=dict(package_versions),
    selected_leiden_resolution=FINAL_LEIDEN_RESOLUTION, selected_run=final_run,
    neighbor_parameters=dict(graph.uns['neighbors']['params']), umap_parameters=dict(UMAP_PARAMETERS),
    figure_contract=dict(source_order=source_order, source_palette=source_palette,
                         dot_row_order=DOT_ROW_ORDER, dot_center=DOT_CENTER,
                         dot_scale=DOT_SCALE, dot_stats=dot_stats, marker_groups=MARKER_GROUPS,
                         marker_colors=MARKER_COLORS, dot_panels=DOT_PANELS, umap_style=umap_style),
    anndata_file=H5AD_PATH.name,
)
DATA_DIR.mkdir(parents=True, exist_ok=True)
joblib.dump(model, MODEL_PATH, compress=3)
with ad.settings.override(allow_write_nullable_strings=True):
    a.write_h5ad(H5AD_PATH, compression='gzip')

# Verify saved results using the fitted reference means without refitting.
saved_model = joblib.load(MODEL_PATH)
saved = ad.read_h5ad(H5AD_PATH)
assert counts_sha256(saved.layers[COUNTS_LAYER], saved.obs_names, saved.var_names) == joint_counts_hash
assert (saved.X != a.X).nnz == 0
assert np.array_equal(saved.obsm['X_css'], a.obsm['X_css'])
assert np.array_equal(saved.obsm['X_umap'], xy)
assert np.array_equal(saved.obsm['css_marker_measured'], marker_data['measured'])
assert np.array_equal(saved.obsm['css_marker_raw_counts'], marker_data['raw'], equal_nan=True)
for key in ('connectivities', 'distances'):
    assert (saved.obsp[key] != graph.obsp[key]).nnz == 0
assert saved.obs.leiden_css.astype(str).equals(a.obs.leiden_css.astype(str))
assert saved.obs.source_stage_celltype.equals(a.obs.source_stage_celltype)
assert not any(c.startswith('css_leiden_') for c in saved.obs)
assert saved_model['selected_run'] == saved.uns['css']['selected_run'] == final_run
assert _css_array_hash(transform_css_reference(saved, saved_model)) == EXPECTED_CSS_SHA256
del saved, saved_model
display(pd.DataFrame([dict(
    cells=a.n_obs, genes=a.n_vars, CSS_dimensions=a.obsm['X_css'].shape[1],
    final_resolution=FINAL_LEIDEN_RESOLUTION, final_clusters=a.obs.leiden_css.nunique(),
    saved_PNG=len(saved_figures), model=MODEL_PATH.name, h5ad=H5AD_PATH.name
)]))


## Reuse saved results

Load `Whole_CSS.h5ad` and `Whole_CSS_model.joblib` from the configured directory.
The H5AD contains original common-axis counts, log1p(CP10k), PCA, CSS, UMAP, the single
Leiden 0.8 partition and the 36 source-annotated categories. The model stores real CSS
reference means, gene/block order and the full/split DotPlot statistics.
The source annotations are retained; they are not newly inferred from Leiden.


# modify

In [ ]:
#18. =========== Shared colors and figure settings
from pathlib import Path
import hashlib
import textwrap

import anndata as ad
import joblib
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.font_manager import fontManager
from matplotlib.lines import Line2D
from matplotlib.transforms import Bbox
from scipy.spatial.distance import cdist
from IPython import get_ipython

get_ipython().run_line_magic("matplotlib", "inline")

ROOT = Path(".")  # Set the project root before running this template.
# Set the stage input directory; uses integration H5ADs generated by the corresponding fig1_E*_clustering.ipynb notebook.
DATA_DIR = ROOT
# Set the output directory path.
OUTPUT_DIR = ROOT
# Set the result file path; uses Whole_CSS_model.joblib generated by whole-period CSS integration.
MODEL_PATH = ROOT
# Set the result file path; uses Whole_CSS.h5ad generated by whole-period CSS integration.
H5AD_PATH = ROOT
# Set the Arial font file path.
FONT_PATH = ROOT

SOURCE_KEY = "source_stage_celltype"
STAGE_KEY = "input_stage"
AXIS_SIZE = 5.6
POINT_SIZE = 1.35
POINT_ALPHA = 0.82
TITLE_FONTSIZE = 10
TITLE_PAD = 10
AXIS_FONTSIZE = 9
AXIS_LABEL_PAD = 5
TICK_FONTSIZE = 7
TICK_PAD = 3
TICK_LENGTH = 3
LABEL_FONTSIZE = 6.75
LABEL_WRAP = 27
LEGEND_FONTSIZE = 7.5
LEGEND_MARKER_SIZE = 5
AXIS_LINEWIDTH = 1.1
LEFT_MARGIN = 0.70
BOTTOM_MARGIN = 0.65
TOP_MARGIN = 0.55
RIGHT_MARGIN = 0.25
LEGEND_SPACE = 2.0
LABEL_SEARCH_STEP = 10
LABEL_MAX_SHIFT = 0.8
CONNECTOR_DISTANCE = 12
CONNECTOR_LINEWIDTH = 0.42
CONNECTOR_ALPHA = 0.70
LABEL_OFFSETS = {}  # Optional offsets in UMAP units, keyed by the full annotation.

CELL_ANNOTATION_COLORS = {
    'E9.5 | CVG neuroblast': '#A05837',
    'E10.5 | O-2: delaminating otic/CVG neuroblast': '#004D43',
    'E10.5 | O-3: developing CVG neuron (not SGN-specific)': '#CB7E98',
    'E11.5 | Undifferentiated / proliferative CVG': '#B903AA',
    'E11.5 | Differentiating SGN': '#006FA6',
    'E13.5 | Late CVG': '#6B7900',
    'E13.5 | Differentiating SGN': '#FF2F80',
    'E14.5 | Type Ia + Type II precursor candidate': '#BA0900',
    'E14.5 | Type Ib + Type Ic precursor candidate': '#A77500',
    'E14.5 | Undifferentiated SGN': '#008941',
    'E15.5 | Type Ia + Type II precursor candidate': '#788D66',
    'E15.5 | Type Ib + Type Ic precursor candidate': '#00A6AA',
    'E15.5 | Undifferentiated SGN': '#6A3A4C',
    'E16.5 | Emerging Type Ic': '#3B9700',
    'E16.5 | Type Ia + Type II precursor candidate': '#FF34FF',
    'E16.5 | Type Ib + Type Ic precursor candidate': '#BC23FF',
    'E16.5 | Undifferentiated SGN': '#0086ED',
    'E17.5 | Type Ib + Type Ic precursor candidate': '#6367A9',
    'E17.5 | Type Ia + Type II precursor candidate': '#7A4900',
    'E18.5 | Immature Type II SGN': '#B79762',
    'E18.5 | Immature Type Ia SGN': '#00846F',
    'E18.5 | Immature Type Ib SGN': '#00489C',
    'E18.5 | Immature Type Ic SGN': '#CC0744',
    'E18.5 | Undifferentiated SGN': '#922329',
    'P1 | Type Ia': '#D16100',
    'P1 | Type Ib': '#A30059',
    'P1 | Type Ic': '#0AA6D8',
    'P1 | Type II': '#324E72',
    'P7 | Type Ia': '#3B5DFF',
    'P7 | Type Ib': '#1E6E00',
    'P7 | Type Ic': '#FF4A46',
    'P7 | Type II': '#9B9700',
    'P28 | Type Ia': '#575329',
    'P28 | Type Ib': '#D157A0',
    'P28 | Type Ic': '#549E79',
    'P28 | Type II': '#72418F',
}

DEVELOPMENTAL_STAGE_COLORS = {
    'E9.5': '#D157A0',
    'E10.5': '#CC0744',
    'E11.5': '#3B9700',
    'E13.5': '#3B5DFF',
    'E14.5': '#D16100',
    'E15.5': '#00846F',
    'E16.5': '#BC23FF',
    'E17.5': '#6A3A4C',
    'E18.5': '#9B9700',
    'P1': '#BA0900',
    'P7': '#006FA6',
    'P28': '#B79762',
}


#19. =========== Load the selected CSS result
model = joblib.load(MODEL_PATH)
a = ad.read_h5ad(H5AD_PATH)
assert np.array_equal(a.obs_names.astype(str), model["cell_ids"]), "Cell order mismatch."
assert np.array_equal(a.var_names.astype(str), model["common_genes"]), "Gene order mismatch."
assert a.uns["css"]["selected_run"] == model["selected_run"] == "h2_r0p4_k10_l0p8"

xy = np.asarray(a.obsm["X_umap"])
digest = hashlib.sha256()
digest.update(str(xy.shape).encode())
digest.update(str(xy.dtype).encode())
digest.update(np.ascontiguousarray(xy).tobytes())
assert digest.hexdigest() == a.uns["css"]["umap_sha256"], "Saved UMAP mismatch."
assert len(model["ordered_genes"]) == 2000
assert model["parameters"]["reference_resolution"] == 0.4
assert model["neighbor_parameters"]["n_neighbors"] == 10
assert model["selected_leiden_resolution"] == 0.8
assert xy.shape == (a.n_obs, 2) and np.isfinite(xy).all()
assert a.obs[[SOURCE_KEY, STAGE_KEY]].notna().all().all()

source_labels = a.obs[SOURCE_KEY].astype(str).to_numpy()
source_order = [str(label) for label in model["figure_contract"]["source_order"]]
stages = a.obs[STAGE_KEY].astype(str).to_numpy()
stage_order = list(DEVELOPMENTAL_STAGE_COLORS)
assert set(source_labels) == set(source_order) == set(CELL_ANNOTATION_COLORS)
assert set(stages) == set(stage_order)
assert all(label.split(" | ", 1)[0] == stage for label, stage in zip(source_labels, stages))

seed = model["figure_contract"]["umap_style"]["random_seed"]
draw_order = np.random.default_rng(seed).permutation(a.n_obs)
center = (xy.min(axis=0) + xy.max(axis=0)) / 2
span = max(float(np.ptp(xy, axis=0).max()) * 1.12, 1.0)
xlim = (center[0] - span / 2, center[0] + span / 2)
ylim = (center[1] - span / 2, center[1] + span / 2)

fontManager.addfont(FONT_PATH)
mpl.rcParams.update({"font.family": "Arial", "font.weight": "normal", "text.usetex": False,
                     "axes.titleweight": "normal", "axes.labelweight": "normal",
                     "axes.unicode_minus": False, "pdf.fonttype": 42})
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

#20. =========== Editable PDF export
from matplotlib.backend_bases import RendererBase
from matplotlib.backends.backend_pdf import PdfFile, RendererPdf, GraphicsContextPdf, Op
from matplotlib.path import Path as MplPath
from matplotlib.transforms import IdentityTransform


class UmapPdfRenderer(RendererBase):
    def __init__(self, width, height):
        super().__init__()
        self.width, self.height = width, height
        self.measure = RendererPdf(None, 72, height / 72, width / 72)
        self.paths = {"Scatter": [], "Axes": []}
        self.texts = []

    def get_canvas_width_height(self):
        return self.width, self.height

    def flipy(self):
        return False

    def get_text_width_height_descent(self, s, prop, ismath):
        return self.measure.get_text_width_height_descent(s, prop, ismath)

    def copy_gc(self, gc):
        copy = self.new_gc()
        copy.copy_properties(gc)
        copy.set_clip_rectangle(None)
        copy.set_clip_path(None)
        return copy

    def draw_path(self, gc, path, transform, rgbFace=None):
        if gc.get_hatch():
            raise ValueError("Hatched paths are not supported by this UMAP exporter.")
        path = path.transformed(transform)
        closed = path.codes is not None and MplPath.CLOSEPOLY in path.codes
        filled = closed and rgbFace is not None and (len(rgbFace) < 4 or rgbFace[3] != 0)
        if not filled:
            rgbFace = None
        group = "Scatter" if filled else "Axes"
        if filled and MplPath.CURVE4 not in path.codes:
            raise ValueError("The scatter group must contain circular vector paths only.")
        if not filled and path.codes is not None:
            allowed = {MplPath.MOVETO, MplPath.LINETO, MplPath.CLOSEPOLY}
            if not set(path.codes).issubset(allowed):
                raise ValueError("Axis and guide lines must be straight vector paths.")
        self.paths[group].append((self.copy_gc(gc), path, rgbFace))

    def draw_text(self, gc, x, y, s, prop, angle, ismath=False, mtext=None):
        if ismath or any(ord(character) > 65535 for character in s):
            raise ValueError("UMAP labels must be plain editable text supported by Arial.")
        if prop.get_style() != "normal" or prop.get_weight() not in ("normal", 400):
            raise ValueError("The shared figure standard uses regular Arial only.")
        self.texts.append((self.copy_gc(gc), x, y, s, prop.copy(), angle))

    def draw_image(self, *args, **kwargs):
        raise ValueError("Images are not allowed in the editable UMAP PDF.")

    def draw_gouraud_triangles(self, *args, **kwargs):
        raise ValueError("Shaded image objects are not allowed in the UMAP PDF.")


def export_editable_umap_pdf(fig, destination):
    """Write three sibling clip groups: scatter, axis paths, and live Arial text."""
    width, height = fig.get_size_inches() * 72
    renderer = UmapPdfRenderer(width, height)
    original_dpi = fig.dpi
    backgrounds = [fig.patch] + [ax.patch for ax in fig.axes]
    visibility = [patch.get_visible() for patch in backgrounds]
    with mpl.rc_context({"pdf.fonttype": 42, "pdf.use14corefonts": False}):
        try:
            fig.set_dpi(72)
            for patch in backgrounds:
                patch.set_visible(False)
            fig.draw(renderer)
        finally:
            fig.set_dpi(original_dpi)
            for patch, visible in zip(backgrounds, visibility):
                patch.set_visible(visible)

        metadata = {"Title": Path(destination).stem.replace("_", " ")} if isinstance(destination, (str, Path)) else {}
        pdf = PdfFile(destination, metadata=metadata)
        try:
            pdf.newPage(width / 72, height / 72)
            writer = RendererPdf(pdf, 72, height / 72, width / 72)
            for group in ("Scatter", "Axes", "Text"):
                pdf.output(Op.gsave, Bbox.from_bounds(0, 0, width, height),
                           Op.rectangle, Op.clip, Op.endpath)
                writer.gc = GraphicsContextPdf(pdf)
                if group == "Text":
                    for args in renderer.texts:
                        writer.draw_text(*args)
                else:
                    for gc, path, face in renderer.paths[group]:
                        writer.draw_path(gc, path, IdentityTransform(), face)
                writer.finalize()
                pdf.output(Op.grestore)
            pdf.finalize()
        finally:
            pdf.close()

#21. =========== UMAP layout
def make_umap_axis(title, labels, palette, with_legend=False):
    right = LEGEND_SPACE if with_legend else RIGHT_MARGIN
    width = LEFT_MARGIN + AXIS_SIZE + right
    height = BOTTOM_MARGIN + AXIS_SIZE + TOP_MARGIN
    fig = plt.figure(figsize=(width, height), dpi=100, facecolor="white")
    ax = fig.add_axes([LEFT_MARGIN / width, BOTTOM_MARGIN / height,
                       AXIS_SIZE / width, AXIS_SIZE / height])
    colors = [palette[label] for label in labels[draw_order]]
    ax.scatter(xy[draw_order, 0], xy[draw_order, 1], c=colors,
               s=POINT_SIZE, alpha=POINT_ALPHA, linewidths=0, edgecolors="none")
    ax.set(xlim=xlim, ylim=ylim)
    ax.set_xlabel("UMAP1", fontsize=AXIS_FONTSIZE, labelpad=AXIS_LABEL_PAD)
    ax.set_ylabel("UMAP2", fontsize=AXIS_FONTSIZE, labelpad=AXIS_LABEL_PAD)
    ax.set_title(title, fontsize=TITLE_FONTSIZE, pad=TITLE_PAD)
    ax.set_box_aspect(1)
    ax.set_facecolor("white")
    ax.grid(False)
    ax.tick_params(labelsize=TICK_FONTSIZE, pad=TICK_PAD, length=TICK_LENGTH,
                   width=0.8, colors="black")
    ax.spines[["top", "right"]].set_visible(False)
    for side in ("left", "bottom"):
        ax.spines[side].set_color("black")
        ax.spines[side].set_linewidth(AXIS_LINEWIDTH)
    return fig, ax


def annotation_anchor(points):
    best_score, best_index = np.inf, 0
    for start in range(0, len(points), 256):
        scores = cdist(points[start:start + 256], points).sum(axis=1)
        local = int(np.argmin(scores))
        if scores[local] < best_score:
            best_score, best_index = scores[local], start + local
    return points[best_index]


def place_annotation_labels(fig, ax):
    anchors = np.asarray([annotation_anchor(xy[source_labels == label]) for label in source_order])
    texts = []
    for label, anchor in zip(source_order, anchors):
        stage, name = label.split(" | ", 1)
        label_text = stage + "\n" + textwrap.fill(name, width=LABEL_WRAP, break_long_words=False)
        texts.append(ax.text(*anchor, label_text, ha="center", va="center", color="black",
                             fontsize=LABEL_FONTSIZE, linespacing=1.1, zorder=5))
    fig.canvas.draw()
    renderer = fig.canvas.get_renderer()
    bounds = ax.get_window_extent(renderer).padded(-3)
    anchors_px = ax.transData.transform(anchors)
    sizes = [(text.get_window_extent(renderer).width + 4,
              text.get_window_extent(renderer).height + 4) for text in texts]
    occupied, placements = [], {}
    order = sorted(range(len(texts)), key=lambda i: -sizes[i][0] * sizes[i][1])
    radii = np.arange(0, LABEL_MAX_SHIFT * max(bounds.width, bounds.height), LABEL_SEARCH_STEP)
    for i in order:
        width, height = sizes[i]
        anchor = anchors_px[i]
        if source_order[i] in LABEL_OFFSETS:
            offset = np.asarray(LABEL_OFFSETS[source_order[i]], dtype=float)
            position = ax.transData.transform(anchors[i] + offset)
            candidates = [(position, Bbox.from_bounds(position[0] - width / 2,
                          position[1] - height / 2, width, height))]
        else:
            candidates = []
            for radius in radii:
                angles = np.linspace(0, 2 * np.pi, max(1, int(2 * np.pi * radius / 12)), endpoint=False)
                for angle in angles:
                    position = anchor + radius * np.array([np.cos(angle), np.sin(angle)])
                    position = np.clip(position, [bounds.x0 + width / 2, bounds.y0 + height / 2],
                                       [bounds.x1 - width / 2, bounds.y1 - height / 2])
                    box = Bbox.from_bounds(position[0] - width / 2, position[1] - height / 2, width, height)
                    if not any(box.overlaps(other) for other in occupied):
                        candidates.append((position, box))
                if candidates:
                    break
        assert candidates, f"Annotation does not fit: {source_order[i]}"
        position, box = min(candidates, key=lambda candidate: np.sum((candidate[0] - anchor) ** 2))
        assert bounds.contains(*box.p0) and bounds.contains(*box.p1), "Annotation is clipped."
        assert not any(box.overlaps(other) for other in occupied), "Annotation offsets overlap."
        placements[i] = position
        occupied.append(box)
        texts[i].set_position(ax.transData.inverted().transform(position))
    for i, text in enumerate(texts):
        delta = anchors_px[i] - placements[i]
        boundary = np.max(np.abs(delta) / (np.asarray(sizes[i]) / 2))
        if np.linalg.norm(delta) > CONNECTOR_DISTANCE and boundary > 1:
            start = placements[i] + delta / boundary
            line = ax.transData.inverted().transform([start, anchors_px[i]])
            ax.plot(line[:, 0], line[:, 1], color="0.32", lw=CONNECTOR_LINEWIDTH,
                    alpha=CONNECTOR_ALPHA, zorder=3)
    return texts


def check_umap_layout(fig, ax, labels=()):
    fig.canvas.draw()
    renderer = fig.canvas.get_renderer()
    axis_box = ax.get_window_extent(renderer)
    assert np.allclose([axis_box.width, axis_box.height], AXIS_SIZE * fig.dpi, atol=0.1)
    assert ax.get_xlim() == tuple(xlim) and ax.get_ylim() == tuple(ylim)
    boxes = [text.get_window_extent(renderer) for text in labels]
    assert not any(left.overlaps(right) for i, left in enumerate(boxes) for right in boxes[i + 1:])
    x_ticks = [tick.label1 for tick in ax.xaxis.get_major_ticks() if xlim[0] <= tick.get_loc() <= xlim[1]]
    y_ticks = [tick.label1 for tick in ax.yaxis.get_major_ticks() if ylim[0] <= tick.get_loc() <= ylim[1]]
    text_objects = [ax.title, ax.xaxis.label, ax.yaxis.label, *x_ticks, *y_ticks, *labels]
    legend = ax.get_legend()
    if legend is not None:
        text_objects.extend(legend.get_texts())
    page = fig.bbox
    for text in text_objects:
        if text.get_visible() and text.get_text():
            box = text.get_window_extent(renderer)
            assert page.contains(*box.p0) and page.contains(*box.p1), f"Text is clipped: {text.get_text()}"
            assert not text.get_path_effects(), "Text must remain editable without outlines."


#22. =========== Cell annotation UMAP
fig, ax = make_umap_axis("Whole period: Cell annotation", source_labels, CELL_ANNOTATION_COLORS)
annotation_texts = place_annotation_labels(fig, ax)
check_umap_layout(fig, ax, annotation_texts)
# Set the output file path.
export_editable_umap_pdf(fig, ROOT)
plt.show()
plt.close(fig)


In [ ]:
#23. =========== Developmental stage UMAP
fig, ax = make_umap_axis("Whole period: Stage distribution", stages,
                        DEVELOPMENTAL_STAGE_COLORS, with_legend=True)
counts = {stage: int(np.count_nonzero(stages == stage)) for stage in stage_order}
handles = [Line2D([], [], marker="o", linestyle="", markersize=LEGEND_MARKER_SIZE,
                  markerfacecolor=DEVELOPMENTAL_STAGE_COLORS[stage], markeredgecolor="none",
                  label=f"{stage} (n={counts[stage]:,})") for stage in stage_order]
ax.legend(handles=handles, loc="center left", bbox_to_anchor=(1.025, 0.5),
          frameon=False, fontsize=LEGEND_FONTSIZE, handletextpad=0.35,
          labelspacing=0.48, borderaxespad=0)
check_umap_layout(fig, ax)
# Set the output file path.
export_editable_umap_pdf(fig, ROOT)
plt.show()
plt.close(fig)


In [ ]:
#24. =========== Cell annotation UMAP with legend
CELL_LEGEND_COLUMNS = 2
CELL_LEGEND_WRAP = 46
CELL_LEGEND_LABEL_SPACING = 0.60
CELL_LEGEND_COLUMN_SPACING = 1.25

source_counts = {label: int(np.count_nonzero(source_labels == label)) for label in source_order}
assert len(source_counts) == 36 and sum(source_counts.values()) == a.n_obs
legend_names = [f"{label} (n={source_counts[label]:,})" for label in source_order]
handles = [Line2D([], [], marker="o", linestyle="", markersize=LEGEND_MARKER_SIZE,
                  markerfacecolor=CELL_ANNOTATION_COLORS[label], markeredgecolor="none",
                  markeredgewidth=0,
                  label=textwrap.fill(name, width=CELL_LEGEND_WRAP,
                                      break_long_words=False, break_on_hyphens=False))
           for label, name in zip(source_order, legend_names)]

fig, ax = make_umap_axis("Whole period: Cell annotation", source_labels,
                        CELL_ANNOTATION_COLORS, with_legend=True)
legend = ax.legend(handles=handles, loc="center left", bbox_to_anchor=(1.025, 0.5),
                   ncols=CELL_LEGEND_COLUMNS, frameon=False, fontsize=LEGEND_FONTSIZE,
                   handletextpad=0.35, labelspacing=CELL_LEGEND_LABEL_SPACING,
                   columnspacing=CELL_LEGEND_COLUMN_SPACING, borderaxespad=0)

# Fit the canvas to the legend while keeping the main axes at 5.6 inches.
fig.canvas.draw()
width = legend.get_window_extent(fig.canvas.get_renderer()).x1 / fig.dpi + RIGHT_MARGIN
height = float(fig.get_size_inches()[1])
fig.set_size_inches(width, height)
ax.set_position([LEFT_MARGIN / width, BOTTOM_MARGIN / height,
                 AXIS_SIZE / width, AXIS_SIZE / height])
check_umap_layout(fig, ax)
legend_texts = legend.get_texts()
assert len(legend_texts) == len(source_order)
assert [" ".join(text.get_text().split()) for text in legend_texts] == legend_names
assert not ax.texts and not ax.lines, "The legend version must not contain annotation labels or connectors."
legend_box = legend.get_window_extent(fig.canvas.get_renderer())
assert fig.bbox.contains(*legend_box.p0) and fig.bbox.contains(*legend_box.p1)
assert legend_box.x0 > ax.get_window_extent().x1, "The legend must remain outside the axes."
# Set the output file path.
export_editable_umap_pdf(fig, ROOT)
plt.show()
plt.close(fig)


In [ ]:
#25. =========== Developmental stage UMAP without legend
fig, ax = make_umap_axis("Whole period: Stage distribution", stages,
                        DEVELOPMENTAL_STAGE_COLORS)
check_umap_layout(fig, ax)
assert ax.get_legend() is None and not ax.texts and not ax.lines
# Set the output file path.
export_editable_umap_pdf(fig, ROOT)
plt.show()
plt.close(fig)


In [ ]:
#26. =========== Saved DotPlot statistics and appearance
from matplotlib.colors import LinearSegmentedColormap, Normalize
from matplotlib.patches import Rectangle

DOT_MAX_AREA = 36
DOT_GAP_RATIO = 0.35  # Edge clearance divided by the maximum data-dot diameter.
DOT_GENE_FONT = 6.3
DOT_TYPE_FONT = 6.5
DOT_GROUP_FONT = 6.5
DOT_GENE_ROTATION = 55
DOT_GROUP_ROTATION = 55
DOT_NARROW_GROUP_ROTATION = 90
DOT_GROUP_WRAP = 22
DOT_GROUP_LINEWIDTH = 1.5
DOT_MARGIN = 0.18
DOT_LABEL_PAD = 2
DOT_COLORBAR_WIDTH = 2.1
DOT_COLORBAR_HEIGHT = 0.075
DOT_COLORBAR_STEPS = 256
DOT_PANEL_NAMES = {
    "dev": ("Development", "Developmental markers"),
    "sgn": ("Spiral_Ganglion", "Spiral ganglion markers"),
    "exclude": ("Exclusion", "Exclusion markers"),
}

dot_contract = model["figure_contract"]
dot_stats = dot_contract["dot_stats"]
dot_levels = np.asarray(dot_contract["dot_row_order"], dtype=str)
dot_genes = np.asarray(a.uns["css_marker_gene_symbols"], dtype=str)
dot_groups = np.asarray(a.uns["css_marker_groups"], dtype=str)
dot_counts = np.asarray(dot_stats["n_cells"], dtype=int)
dot_pct = np.asarray(dot_stats["pct"], dtype=float)
dot_coverage = np.asarray(dot_stats["coverage"], dtype=float)
dot_mean = np.asarray(dot_stats["mean"], dtype=float)
dot_center = np.asarray(dot_contract["dot_center"], dtype=float)
dot_scale = np.asarray(dot_contract["dot_scale"], dtype=float)
dot_scaled = np.divide(dot_mean - dot_center, dot_scale,
                       out=np.zeros_like(dot_mean), where=dot_scale > 0)
dot_scaled[dot_coverage == 0] = np.nan
dot_marker_colors = dot_contract["marker_colors"]
dot_panels = [("Full", "Marker expression", np.arange(len(dot_genes)))]
for key, groups in dot_contract["dot_panels"]:
    name, title = DOT_PANEL_NAMES[key]
    dot_panels.append((name, title, np.flatnonzero(np.isin(dot_groups, groups))))

assert len(dot_levels) == 36 and set(dot_levels) == set(source_order)
assert np.array_equal(dot_levels, np.asarray(dot_stats["levels"], dtype=str))
assert np.array_equal(dot_counts, [np.count_nonzero(source_labels == label) for label in dot_levels])
assert len(set(dot_genes)) == len(dot_genes) == 168
assert dot_pct.shape == dot_scaled.shape == dot_coverage.shape == (36, 168)
assert np.array_equal(np.concatenate([cols for _, _, cols in dot_panels[1:]]), dot_panels[0][2])
assert np.isfinite(dot_pct[dot_coverage > 0]).all()
assert np.isfinite(dot_scaled[dot_coverage > 0]).all()
assert ((dot_pct[dot_coverage > 0] >= 0) & (dot_pct[dot_coverage > 0] <= 100)).all()
assert np.allclose(dot_center, np.nanmean(dot_mean, axis=0))
assert np.allclose(dot_scale, np.nanstd(dot_mean, axis=0, ddof=0))
dot_diameter = np.sqrt(DOT_MAX_AREA * np.nanmax(dot_pct) / 100)
dot_pitch = (1 + DOT_GAP_RATIO) * dot_diameter / 72
dot_cmap = LinearSegmentedColormap.from_list(
    "marker_purple", ["#F1F1F1", "#E8C6DE", "#C67ABA", "#952E7D", "#69005F"])
dot_norm = Normalize(-1, 1, clip=True)


In [ ]:
#27. =========== Editable DotPlot PDF export
class DotplotPdfRenderer(UmapPdfRenderer):
    def __init__(self, width, height):
        super().__init__(width, height)
        self.artist_groups = []

    def open_group(self, name, gid=None):
        self.artist_groups.append(gid)

    def close_group(self, name):
        self.artist_groups.pop()

    def draw_path(self, gc, path, transform, rgbFace=None):
        closed = path.codes is not None and MplPath.CLOSEPOLY in path.codes
        filled = closed and rgbFace is not None and (len(rgbFace) < 4 or rgbFace[3] != 0)
        curved = path.codes is not None and MplPath.CURVE4 in path.codes
        if filled and not curved:
            assert "dotplot-colorbar" in self.artist_groups, "Only colorbar rectangles may be filled."
            path = path.transformed(transform)
            assert list(path.codes) == [MplPath.MOVETO, MplPath.LINETO, MplPath.LINETO,
                                        MplPath.LINETO, MplPath.CLOSEPOLY]
            vertices = path.vertices[:4]
            assert len(np.unique(vertices[:, 0])) == len(np.unique(vertices[:, 1])) == 2
            self.paths["Axes"].append((self.copy_gc(gc), path, rgbFace))
        else:
            super().draw_path(gc, path, transform, rgbFace)

    def draw_quad_mesh(self, *args, **kwargs):
        raise ValueError("Use independent vector rectangles for the colorbar.")


def export_editable_dotplot_pdf(fig, destination):
    """Keep circles, guide paths and colorbar swatches, and Arial text in three sibling clips."""
    width, height = fig.get_size_inches() * 72
    renderer = DotplotPdfRenderer(width, height)
    original_dpi = fig.dpi
    backgrounds = [fig.patch] + [ax.patch for ax in fig.axes]
    visibility = [patch.get_visible() for patch in backgrounds]
    with mpl.rc_context({"pdf.fonttype": 42, "pdf.use14corefonts": False}):
        try:
            fig.set_dpi(72)
            for patch in backgrounds:
                patch.set_visible(False)
            fig.draw(renderer)
        finally:
            fig.set_dpi(original_dpi)
            for patch, visible in zip(backgrounds, visibility):
                patch.set_visible(visible)
        pdf = PdfFile(destination, metadata={"Title": Path(destination).stem.replace("_", " ")})
        try:
            pdf.newPage(width / 72, height / 72)
            writer = RendererPdf(pdf, 72, height / 72, width / 72)
            for group in ("Scatter", "Axes", "Text"):
                pdf.output(Op.gsave, Bbox.from_bounds(0, 0, width, height),
                           Op.rectangle, Op.clip, Op.endpath)
                writer.gc = GraphicsContextPdf(pdf)
                if group == "Text":
                    for args in renderer.texts:
                        writer.draw_text(*args)
                else:
                    for gc, path, face in renderer.paths[group]:
                        writer.draw_path(gc, path, IdentityTransform(), face)
                writer.finalize()
                pdf.output(Op.grestore)
            pdf.finalize()
        finally:
            pdf.close()


#28. =========== Full and split DotPlots
def save_article_dotplot(name, title, columns):
    genes, groups = dot_genes[columns], dot_groups[columns]
    pct, coverage, scaled = dot_pct[:, columns], dot_coverage[:, columns], dot_scaled[:, columns]
    n_rows, n_genes = pct.shape
    matrix_width, matrix_height = n_genes * dot_pitch, n_rows * dot_pitch
    left, right, bottom, top = 2.5, DOT_MARGIN, 0.8, 2.0
    width, height = left + matrix_width + right, bottom + matrix_height + top
    fig = plt.figure(figsize=(width, height), dpi=100, facecolor="white")
    try:
        ax = fig.add_axes([left / width, bottom / height,
                           matrix_width / width, matrix_height / height])
        xx, yy = np.meshgrid(np.arange(n_genes), np.arange(n_rows))
        positive = (coverage > 0) & (pct > 0)
        points = ax.scatter(xx[positive], yy[positive], s=DOT_MAX_AREA * pct[positive] / 100,
                            c=scaled[positive], cmap=dot_cmap, norm=dot_norm,
                            edgecolors="none", linewidths=0)
        missing = coverage == 0
        if missing.any():
            ax.scatter(xx[missing], yy[missing], marker="x", s=DOT_MAX_AREA * 0.30,
                       c="0.67", linewidths=0.4)
        partial = np.any(coverage < 1, axis=0)
        gene_names = [gene + ("\u2020" if flag else "") for gene, flag in zip(genes, partial)]
        ax.set_xticks(np.arange(n_genes), gene_names, rotation=DOT_GENE_ROTATION,
                      ha="right", fontsize=DOT_GENE_FONT)
        ax.set_yticks(np.arange(n_rows), [f"{label} (n={count:,})"
                      for label, count in zip(dot_levels, dot_counts)], fontsize=DOT_TYPE_FONT)
        ax.set(xlim=(-0.5, n_genes - 0.5), ylim=(n_rows - 0.5, -0.5))
        ax.tick_params(length=0, pad=DOT_LABEL_PAD)
        ax.set_facecolor("white")
        ax.grid(False)
        for spine in ax.spines.values():
            spine.set_visible(False)

        starts = np.r_[0, np.flatnonzero(groups[1:] != groups[:-1]) + 1]
        ends = np.r_[starts[1:], n_genes]
        assert len(set(groups)) == len(starts), "Marker groups must be contiguous."
        headings = []
        for start, end in zip(starts, ends):
            ax.plot([start - 0.5 + 0.08, end - 0.5 - 0.08], [-0.90, -0.90],
                    color=dot_marker_colors[groups[start]], lw=DOT_GROUP_LINEWIDTH,
                    clip_on=False, solid_capstyle="butt")
            narrow = end - start < 4
            heading = groups[start] if narrow else textwrap.fill(
                groups[start], width=DOT_GROUP_WRAP, break_long_words=False, break_on_hyphens=False)
            headings.append(ax.text((start + end - 1) / 2, -1.4, heading, ha="center", va="bottom",
                                    rotation=DOT_NARROW_GROUP_ROTATION if narrow else DOT_GROUP_ROTATION,
                                    fontsize=DOT_GROUP_FONT,
                                    linespacing=1.1, clip_on=False))

        # Move only colliding group headings; retain the fixed data-dot pitch.
        fig.canvas.draw()
        renderer = fig.canvas.get_renderer()
        occupied = []
        for text in headings:
            for attempt in range(150):
                box = text.get_window_extent(renderer).padded(1.5)
                if not any(box.overlaps(other) for other in occupied):
                    break
                x, y = text.get_position()
                text.set_position((x, y - 0.25))
            else:
                raise ValueError("Marker group headings do not fit.")
            occupied.append(box)

        fig.canvas.draw()
        renderer = fig.canvas.get_renderer()
        matrix_box = ax.get_window_extent(renderer)
        row_boxes = [text.get_window_extent(renderer) for text in ax.get_yticklabels()]
        gene_boxes = [text.get_window_extent(renderer) for text in ax.get_xticklabels()]
        heading_boxes = [text.get_window_extent(renderer) for text in headings]
        left = max(box.width for box in row_boxes) / fig.dpi + DOT_MARGIN
        left = max(left, (matrix_box.x0 - min(box.x0 for box in heading_boxes)) / fig.dpi + DOT_MARGIN)
        right = max(DOT_MARGIN, (max(box.x1 for box in heading_boxes) - matrix_box.x1) / fig.dpi + DOT_MARGIN)
        bottom = (matrix_box.y0 - min(box.y0 for box in gene_boxes)) / fig.dpi + 0.62
        heading_height = (max(box.y1 for box in heading_boxes) - matrix_box.y1) / fig.dpi
        top = heading_height + 1.0
        width, height = left + matrix_width + right, bottom + matrix_height + top
        fig.set_size_inches(width, height)
        ax.set_position([left / width, bottom / height,
                         matrix_width / width, matrix_height / height])
        fig.text((left + matrix_width / 2) / width, (height - 0.16) / height,
                 "Whole period: " + title, ha="center", va="top", fontsize=TITLE_FONTSIZE)

        # Independent vector swatches avoid rasterized colorbars and nested clips.
        legend_y = bottom + matrix_height + heading_height + 0.30
        cax = fig.add_axes([left / width, legend_y / height,
                           DOT_COLORBAR_WIDTH / width, DOT_COLORBAR_HEIGHT / height])
        edges = np.linspace(-1, 1, DOT_COLORBAR_STEPS + 1)
        for lo, hi in zip(edges[:-1], edges[1:]):
            cax.add_patch(Rectangle((lo, 0), hi - lo, 1,
                                   facecolor=dot_cmap(dot_norm((lo + hi) / 2)),
                                   edgecolor="none", linewidth=0, gid="dotplot-colorbar"))
        cax.set(xlim=(-1, 1), ylim=(0, 1))
        cax.set_xticks([-1, -0.5, 0, 0.5, 1], ["-1.0", "-0.5", "0.0", "0.5", "1.0"])
        cax.set_yticks([])
        cax.tick_params(labelsize=TICK_FONTSIZE, length=2, pad=2)
        cax.set_title("Avg. Exp.", fontsize=LEGEND_FONTSIZE, loc="left", pad=5)
        for spine in cax.spines.values():
            spine.set_visible(False)
        handles = [Line2D([], [], marker="o", linestyle="", markerfacecolor="0.5",
                          markeredgecolor="none", markeredgewidth=0,
                          markersize=np.sqrt(DOT_MAX_AREA * value / 100))
                   for value in (0, 25, 50, 75, 100)]
        legend = fig.legend(handles, ["0", "25", "50", "75", "100"], title="% Exp.",
                            loc="center left", bbox_to_anchor=((left + DOT_COLORBAR_WIDTH + 0.35) / width,
                                                              (legend_y + DOT_COLORBAR_HEIGHT / 2) / height),
                            frameon=False, ncol=5, fontsize=LEGEND_FONTSIZE,
                            title_fontsize=LEGEND_FONTSIZE, handletextpad=0.35,
                            columnspacing=0.8, handlelength=0.9, borderaxespad=0)
        notes = [
            "Color: gene-wise z-score across all 36 annotated types (ddof=0); shared scaling.",
            "\u2020 Not measured in every source; % Exp. among measured cells. \u00d7 No measured cells.",
        ]
        for i, note in enumerate(notes):
            fig.text(left / width, (0.08 + (1 - i) * 0.12) / height,
                     note, fontsize=5.8, ha="left", va="bottom")

        fig.canvas.draw()
        renderer = fig.canvas.get_renderer()
        diameter_px = points.get_paths()[0].get_extents().width * points.get_transforms()[:, 0, 0].max()
        centers = ax.transData.transform([[0, 0], [1, 0], [0, 1]])
        pitch_x, pitch_y = abs(centers[1, 0] - centers[0, 0]), abs(centers[2, 1] - centers[0, 1])
        assert np.isclose(diameter_px * 72 / fig.dpi, dot_diameter)
        assert np.allclose(np.array([pitch_x, pitch_y]) / diameter_px - 1, DOT_GAP_RATIO, atol=0.005)
        assert np.allclose(points.get_sizes(), DOT_MAX_AREA * pct[positive] / 100)
        assert np.allclose(points.get_facecolors(), dot_cmap(dot_norm(scaled[positive])))
        text_objects = [*ax.get_xticklabels(), *ax.get_yticklabels(), *headings,
                        *cax.get_xticklabels(), cax._left_title, *legend.get_texts(),
                        legend.get_title(), *fig.texts]
        for text in text_objects:
            box = text.get_window_extent(renderer)
            assert fig.bbox.contains(*box.p0) and fig.bbox.contains(*box.p1), f"Text is clipped: {text.get_text()}"
            assert not text.get_path_effects()
        heading_boxes = [text.get_window_extent(renderer) for text in headings]
        assert not any(a.overlaps(b) for i, a in enumerate(heading_boxes) for b in heading_boxes[i + 1:])
        row_boxes = [text.get_window_extent(renderer) for text in ax.get_yticklabels()]
        assert not any(a.overlaps(b) for i, a in enumerate(row_boxes) for b in row_boxes[i + 1:])
        # Set the output file path.
        export_editable_dotplot_pdf(fig, ROOT)
    finally:
        plt.close(fig)


for name, title, columns in dot_panels:
    save_article_dotplot(name, title, columns)
